In [1]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from keras.callbacks import EarlyStopping, LearningRateScheduler

In [2]:
# Upload the padding data

X_train_padded = np.load(
    "/kaggle/input/datasets/logeshm0324/dataset-data/X_train_padded.npy"
)

X_val_padded = np.load(
    "/kaggle/input/datasets/logeshm0324/dataset-data/X_val_padded.npy"
)

X_test_padded = np.load(
    "/kaggle/input/datasets/logeshm0324/dataset-data/X_test_padded.npy"
)

y_train = np.load(
    "/kaggle/input/datasets/logeshm0324/dataset-data/y_train.npy"
)

y_val = np.load(
    "/kaggle/input/datasets/logeshm0324/dataset-data/y_val.npy"
)

y_test = np.load(
    "/kaggle/input/datasets/logeshm0324/dataset-data/y_test.npy"
)

In [4]:
with open(
    "/kaggle/input/datasets/logeshm0324/dataset-data/feature_config.pkl",
    "rb"
) as file:

    feature_config = pickle.load(file)

In [5]:

VOCAB_SIZE = feature_config["vocab_size"]
EMBEDDING_DIM = feature_config["embedding_dim"]
MAX_SEQUENCE_LENGTH = feature_config["max_sequence_length"]

print("Vocabulary size:", VOCAB_SIZE)
print("Embedding dimension:", EMBEDDING_DIM)
print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)

Vocabulary size: 20000
Embedding dimension: 128
Maximum sequence length: 200


In [7]:
# Load TF-IDF Features for ANN

with open(
    "/kaggle/input/datasets/logeshm0324/dataset-data/tfidf_vectorizer.pkl",
    "rb"
) as file:

    tfidf_vectorizer = pickle.load(file)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
with open(
    "/kaggle/input/datasets/logeshm0324/dataset-data/text_splits.pkl",
    "rb"
) as file:

    text_splits = pickle.load(file)

In [9]:
X_train_text = text_splits["X_train"]
X_val_text = text_splits["X_val"]
X_test_text = text_splits["X_test"]

In [10]:
# Fit already fitted TF-IDF vectorizer

X_train_tfidf = tfidf_vectorizer.transform(
    X_train_text
)

X_val_tfidf = tfidf_vectorizer.transform(
    X_val_text
)

X_test_tfidf = tfidf_vectorizer.transform(
    X_test_text
)

In [11]:
print("TF-IDF train:", X_train_tfidf.shape)
print("TF-IDF validation:", X_val_tfidf.shape)
print("TF-IDF test:", X_test_tfidf.shape)

TF-IDF train: (34705, 20000)
TF-IDF validation: (7439, 20000)
TF-IDF test: (7438, 20000)


In [83]:
optimization_results = []

# ANN

In [ ]:
TFIDF_FEATURES = X_train_tfidf.shape[1]

In [ ]:
# Dropout

ann_model = Sequential([

    Input(shape=(TFIDF_FEATURES,)),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),


    Dense(
        1,
        activation="sigmoid"
    )
])

E0000 00:00:1787561036.230209  215937 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size=64
)

W0000 00:00:1787561069.034392  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.8692 - loss: 0.3333 - val_accuracy: 0.9032 - val_loss: 0.2430
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 21s 39ms/step - accuracy: 0.9457 - loss: 0.1507 - val_accuracy: 0.8961 - val_loss: 0.2684
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 21s 38ms/step - accuracy: 0.9759 - loss: 0.0759 - val_accuracy: 0.8896 - val_loss: 0.3467
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.9891 - loss: 0.0374 - val_accuracy: 0.8919 - val_loss: 0.4082
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 23s 42ms/step - accuracy: 0.9956 - loss: 0.0165 - val_accuracy: 0.8892 - val_loss: 0.5153
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 22s 41ms/step - accuracy: 0.9971 - loss: 0.0096 - val_accuracy: 0.8907 - val_loss: 0.5869
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - accuracy: 0.9981 - loss: 0.0061 - val_accuracy: 0.8887 - val_loss: 0.6611
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.9985 - loss: 0.0040 - 

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [12]:
def evaluate_model(y_true, y_pred, y_prob):

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

In [ ]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [ ]:
ann_metrics

{'Accuracy': 0.8827641839204087,
 'Precision': 0.9001398601398601,
 'Recall': 0.8620412536833646,
 'F1 Score': 0.8806787082649151,
 'ROC-AUC': 0.953827102116188}

In [84]:
optimization_results.append({
    "Experiment": "E1",
    "Enhancement": "DropOut 0.3",
    "Model": "ANN",
    "Accuracy": 0.8827641839204087,
    "Precision": 0.9001398601398601,
    "Recall": 0.8620412536833646,
    "F1 Score": 0.8806787082649151,
    "ROC-AUC": 0.953827102116188
})

In [ ]:
# Dropout 0.5

ann_model = Sequential([

    Input(shape=(TFIDF_FEATURES,)),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),


    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size=64
)

W0000 00:00:1787561654.203858  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 28s 46ms/step - accuracy: 0.8525 - loss: 0.3633 - val_accuracy: 0.9042 - val_loss: 0.2405
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 37s 38ms/step - accuracy: 0.9380 - loss: 0.1766 - val_accuracy: 0.9040 - val_loss: 0.2510
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 37s 31ms/step - accuracy: 0.9634 - loss: 0.1124 - val_accuracy: 0.8986 - val_loss: 0.2899
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.9775 - loss: 0.0718 - val_accuracy: 0.8990 - val_loss: 0.3430
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9858 - loss: 0.0452 - val_accuracy: 0.8958 - val_loss: 0.4001
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 30ms/step - accuracy: 0.9901 - loss: 0.0316 - val_accuracy: 0.8949 - val_loss: 0.4313
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 36ms/step - accuracy: 0.9933 - loss: 0.0213 - val_accuracy: 0.8949 - val_loss: 0.4992
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 30ms/step - accuracy: 0.9941 - loss: 0.0182 - 

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [ ]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [ ]:
ann_metrics

{'Accuracy': 0.8822264049475665,
 'Precision': 0.8897680763983629,
 'Recall': 0.8735601392981516,
 'F1 Score': 0.8815896188158961,
 'ROC-AUC': 0.9551873667147117}

In [85]:
optimization_results.append({
    "Experiment": "E2",
    "Enhancement": "DropOut 0.5",
    "Model": "ANN",
    "Accuracy": 0.8822264049475665,
    "Precision": 0.8897680763983629,
    "Recall": 0.8735601392981516,
    "F1 Score": 0.8815896188158961,
    "ROC-AUC": 0.9551873667147117
})

In [ ]:
# Batchnormalization

ann_model = Sequential([

    Input(shape=(TFIDF_FEATURES,)),

    Dense( 64 ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),

    tf.keras.layers.Dropout(0.5),

    Dense(32),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size=64
)

W0000 00:00:1787562669.683050  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 23s 35ms/step - accuracy: 0.8286 - loss: 0.3829 - val_accuracy: 0.8997 - val_loss: 0.2743
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9234 - loss: 0.2066 - val_accuracy: 0.8968 - val_loss: 0.2568
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - accuracy: 0.9521 - loss: 0.1362 - val_accuracy: 0.8973 - val_loss: 0.3021
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9665 - loss: 0.0960 - val_accuracy: 0.8939 - val_loss: 0.3434
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9742 - loss: 0.0730 - val_accuracy: 0.8934 - val_loss: 0.3761
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - accuracy: 0.9777 - loss: 0.0638 - val_accuracy: 0.8914 - val_loss: 0.3860
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - accuracy: 0.9820 - loss: 0.0534 - val_accuracy: 0.8903 - val_loss: 0.4145
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - accuracy: 0.9851 - loss: 0.0449 - 

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [ ]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [ ]:
ann_metrics

{'Accuracy': 0.8822264049475665,
 'Precision': 0.8897680763983629,
 'Recall': 0.8735601392981516,
 'F1 Score': 0.8815896188158961,
 'ROC-AUC': 0.9544096801586897}

In [86]:
optimization_results.append({
    "Experiment": "E3",
    "Enhancement": "BatchNormalization",
    "Model": "ANN",
    "Accuracy": 0.8822264049475665,
    "Precision": 0.8897680763983629,
    "Recall": 0.8735601392981516,
    "F1 Score": 0.8815896188158961,
    "ROC-AUC": 0.9544096801586897
})

In [ ]:
# BatchNormalization without dropout

ann_model = Sequential([

    Input(shape=(TFIDF_FEATURES,)),

    Dense( 64 ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),

    Dense(32),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Activation("relu"),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size=64
)

W0000 00:00:1787562950.233311  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 25s 38ms/step - accuracy: 0.8707 - loss: 0.3048 - val_accuracy: 0.8931 - val_loss: 0.2656
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9687 - loss: 0.0897 - val_accuracy: 0.8868 - val_loss: 0.3196
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 32ms/step - accuracy: 0.9897 - loss: 0.0338 - val_accuracy: 0.8773 - val_loss: 0.4074
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - accuracy: 0.9927 - loss: 0.0229 - val_accuracy: 0.8851 - val_loss: 0.4526
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - accuracy: 0.9939 - loss: 0.0190 - val_accuracy: 0.8822 - val_loss: 0.4552
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 34ms/step - accuracy: 0.9959 - loss: 0.0123 - val_accuracy: 0.8814 - val_loss: 0.5147
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 34ms/step - accuracy: 0.9972 - loss: 0.0097 - val_accuracy: 0.8816 - val_loss: 0.5481
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9976 - loss: 0.0080 - 

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [ ]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [ ]:
ann_metrics

{'Accuracy': 0.8822264049475665,
 'Precision': 0.8897680763983629,
 'Recall': 0.8735601392981516,
 'F1 Score': 0.8815896188158961,
 'ROC-AUC': 0.9476206847560493}

In [ ]:
# New Optimizers

ann_model = Sequential([

    Input(shape=(TFIDF_FEATURES,)),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),


    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
ann_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size=64
)

W0000 00:00:1787563444.736503  215937 cpu_allocator_impl.cc:82] Allocation of 57630928 exceeds 10% of free system memory.


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.8327 - loss: 0.4069 - val_accuracy: 0.9021 - val_loss: 0.2422
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.9213 - loss: 0.2188 - val_accuracy: 0.9079 - val_loss: 0.2383
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9398 - loss: 0.1762 - val_accuracy: 0.9052 - val_loss: 0.2521
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.9523 - loss: 0.1468 - val_accuracy: 0.9039 - val_loss: 0.2762
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.9599 - loss: 0.1265 - val_accuracy: 0.9004 - val_loss: 0.2982
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9667 - loss: 0.1112 - val_accuracy: 0.8977 - val_loss: 0.3319
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9726 - loss: 0.0970 - val_accuracy: 0.8946 - val_loss: 0.3599
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9751 - loss: 0.0869 - 

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [ ]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [ ]:
ann_metrics

{'Accuracy': 0.8822264049475665,
 'Precision': 0.8897680763983629,
 'Recall': 0.8735601392981516,
 'F1 Score': 0.8815896188158961,
 'ROC-AUC': 0.9543755533406866}

In [87]:
optimization_results.append({
    "Experiment": "E4",
    "Enhancement": "Optimizers RMSprop",
    "Model": "ANN",
    "Accuracy": 0.8822264049475665,
    "Precision": 0.8897680763983629,
    "Recall": 0.8735601392981516,
    "F1 Score": 0.8815896188158961,
    "ROC-AUC": 0.9543755533406866
})

In [ ]:
# Learning Rate

ann_model = Sequential([

    Input(shape=(TFIDF_FEATURES,)),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activaation="relu"
    ),

    tf.keras.layers.Dropout(0.5),


    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
ann_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 24ms/step - accuracy: 0.6187 - loss: 0.6803 - val_accuracy: 0.7816 - val_loss: 0.6496
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.7913 - loss: 0.6015 - val_accuracy: 0.8693 - val_loss: 0.5235
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.8464 - loss: 0.4717 - val_accuracy: 0.8868 - val_loss: 0.3851
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.8751 - loss: 0.3616 - val_accuracy: 0.8950 - val_loss: 0.3023
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.8909 - loss: 0.3007 - val_accuracy: 0.9004 - val_loss: 0.2649
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.9023 - loss: 0.2675 - val_accuracy: 0.9042 - val_loss: 0.2480
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - accuracy: 0.9130 - loss: 0.2403 - val_accuracy: 0.9050 - val_loss: 0.2395
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - accuracy: 0.9191 - loss: 0.2247 - 

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [ ]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [ ]:
ann_metrics

{'Accuracy': 0.9046786770637268,
 'Precision': 0.9044943820224719,
 'Recall': 0.9057058665952317,
 'F1 Score': 0.9050997189131308,
 'ROC-AUC': 0.9674303626733588}

In [88]:
optimization_results.append({
    "Experiment": "E5",
    "Enhancement": "Learning Rate Rmsprop(0.0001)",
    "Model": "ANN",
    'Accuracy': 0.9046786770637268,
    'Precision': 0.9044943820224719,
    'Recall': 0.9057058665952317,
    'F1 Score': 0.9050997189131308,
    'ROC-AUC': 0.9674303626733588
})

In [ ]:
ann_model.save(
    "../models/ann/ann_baseline_5_90.keras"
)

In [ ]:
with open(
    "../models/ann/ann_history_5_90.pkl",
    "wb"
) as file:

    pickle.dump(
        ann_history.history,
        file
    )

In [ ]:
# Adam

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 27ms/step - accuracy: 0.9321 - loss: 0.1900 - val_accuracy: 0.9103 - val_loss: 0.2321
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.9428 - loss: 0.1699 - val_accuracy: 0.9097 - val_loss: 0.2332
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9514 - loss: 0.1503 - val_accuracy: 0.9099 - val_loss: 0.2371
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.9591 - loss: 0.1338 - val_accuracy: 0.9087 - val_loss: 0.2425
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.9658 - loss: 0.1149 - val_accuracy: 0.9094 - val_loss: 0.2503
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9721 - loss: 0.0995 - val_accuracy: 0.9090 - val_loss: 0.2598
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9761 - loss: 0.0874 - val_accuracy: 0.9076 - val_loss: 0.2712
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.9817 - loss: 0.0738 - 

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [ ]:
ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

In [ ]:
ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

In [ ]:
ann_metrics

{'Accuracy': 0.8962086582414628,
 'Precision': 0.8978769148078474,
 'Recall': 0.8949906241628717,
 'F1 Score': 0.8964314462033808,
 'ROC-AUC': 0.9634899443378584}

In [89]:
optimization_results.append({
    "Experiment": "E6",
    "Enhancement": "Learning Rate Adam(0.0001)",
    "Model": "ANN",
    'Accuracy': 0.8962086582414628,
    'Precision': 0.8978769148078474,
    'Recall': 0.8949906241628717,
    'F1 Score': 0.8964314462033808,
    'ROC-AUC': 0.9634899443378584
})

In [ ]:
# Adam

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.00025),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.9855 - loss: 0.0531 - val_accuracy: 0.9008 - val_loss: 0.3417
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 31ms/step - accuracy: 0.9909 - loss: 0.0385 - val_accuracy: 0.9001 - val_loss: 0.3862
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9929 - loss: 0.0286 - val_accuracy: 0.8994 - val_loss: 0.4246
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.9950 - loss: 0.0207 - val_accuracy: 0.8951 - val_loss: 0.4564
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9966 - loss: 0.0157 - val_accuracy: 0.8935 - val_loss: 0.5065
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.9976 - loss: 0.0111 - val_accuracy: 0.8934 - val_loss: 0.5351
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9978 - loss: 0.0099 - val_accuracy: 0.8921 - val_loss: 0.5708
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.9980 - loss: 0.0080 - 

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


{'Accuracy': 0.8881419736488303,
 'Precision': 0.8902340597255851,
 'Recall': 0.8864184302169836,
 'F1 Score': 0.8883221476510067,
 'ROC-AUC': 0.9565075756836299}

In [90]:
optimization_results.append({
    "Experiment": "E7",
    "Enhancement": "Learning Rate Adam(0.00025)",
    "Model": "ANN",
    'Accuracy': 0.8881419736488303,
    'Precision': 0.8902340597255851,
    'Recall': 0.8864184302169836,
    'F1 Score': 0.8883221476510067,
    'ROC-AUC': 0.9565075756836299
})

In [ ]:
# Adam

ann_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate = 0.00025),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 26ms/step - accuracy: 0.9991 - loss: 0.0042 - val_accuracy: 0.8942 - val_loss: 0.7272
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.9990 - loss: 0.0041 - val_accuracy: 0.8939 - val_loss: 0.7500
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.9993 - loss: 0.0032 - val_accuracy: 0.8934 - val_loss: 0.7853
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.9993 - loss: 0.0031 - val_accuracy: 0.8937 - val_loss: 0.7921
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.9993 - loss: 0.0029 - val_accuracy: 0.8919 - val_loss: 0.8180
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9993 - loss: 0.0026 - val_accuracy: 0.8941 - val_loss: 0.8322
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9993 - loss: 0.0023 - val_accuracy: 0.8943 - val_loss: 0.8448
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9993 - loss: 0.0024 - 

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


{'Accuracy': 0.8855875235278301,
 'Precision': 0.8824309978768577,
 'Recall': 0.8907045271899277,
 'F1 Score': 0.886548460205306,
 'ROC-AUC': 0.9553278144773626}

In [91]:
optimization_results.append({
    "Experiment": "E8",
    "Enhancement": "Learning Rate Rmsprop(0.00025)",
    "Model": "ANN",
    'Accuracy': 0.8855875235278301,
    'Precision': 0.8824309978768577,
    'Recall': 0.8907045271899277,
    'F1 Score': 0.886548460205306,
    'ROC-AUC': 0.9553278144773626
})

In [ ]:
# Batch Size Tuning

# Learning Rate

ann_model = Sequential([

    Input(shape=(TFIDF_FEATURES,)),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),


    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
ann_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size= 32
)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - accuracy: 0.6724 - loss: 0.6701 - val_accuracy: 0.8646 - val_loss: 0.6063
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.8332 - loss: 0.5113 - val_accuracy: 0.8812 - val_loss: 0.3841
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.8724 - loss: 0.3500 - val_accuracy: 0.8931 - val_loss: 0.2821
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.8915 - loss: 0.2821 - val_accuracy: 0.9021 - val_loss: 0.2514
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 19ms/step - accuracy: 0.9066 - loss: 0.2502 - val_accuracy: 0.9054 - val_loss: 0.2409
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9142 - loss: 0.2315 - val_accuracy: 0.9064 - val_loss: 0.2362
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - accuracy: 0.9202 - loss: 0.2184 - val_accuracy: 0.9071 - val_loss: 0.2351
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9256 -

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


{'Accuracy': 0.9049475665501479,
 'Precision': 0.9058476394849786,
 'Recall': 0.9046343423519957,
 'F1 Score': 0.9052405843720681,
 'ROC-AUC': 0.9672767919923446}

In [92]:
optimization_results.append({
    "Experiment": "E9",
    "Enhancement": "Batch Size 32",
    "Model": "ANN",
    'Accuracy': 0.9049475665501479,
    'Precision': 0.9058476394849786,
    'Recall': 0.9046343423519957,
    'F1 Score': 0.9052405843720681,
    'ROC-AUC': 0.9672767919923446
})

In [ ]:
# Early Stopping

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=20,
    batch_size= 64,

    callbacks=[
        early_stopping
    ]
)

Epoch 1/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9381 - loss: 0.1794 - val_accuracy: 0.9086 - val_loss: 0.2381
Epoch 2/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9402 - loss: 0.1753 - val_accuracy: 0.9078 - val_loss: 0.2389
Epoch 3/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9427 - loss: 0.1666 - val_accuracy: 0.9074 - val_loss: 0.2409


In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.905216456036569,
 'Precision': 0.9007411328745368,
 'Recall': 0.9115992499330298,
 'F1 Score': 0.9061376647583544,
 'ROC-AUC': 0.9672536551665798}

In [93]:
optimization_results.append({
    "Experiment": "E10",
    "Enhancement": "Early Stopping",
    "Model": "ANN",
    'Accuracy': 0.905216456036569,
    'Precision': 0.9007411328745368,
    'Recall': 0.9115992499330298,
    'F1 Score': 0.9061376647583544,
    'ROC-AUC': 0.9672536551665798
})

In [ ]:
# LR Schedule

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
ann_history = ann_model.fit(
    X_train_tfidf,
    y_train,

    validation_data=(
        X_val_tfidf,
        y_val
    ),

    epochs=10,
    batch_size= 64,

    callbacks=[
        lr_scheduler
    ]
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9396 - loss: 0.1746 - val_accuracy: 0.9075 - val_loss: 0.2395 - learning_rate: 1.0000e-04
Epoch 2/10
542/543 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9421 - loss: 0.1695
Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.9421 - loss: 0.1695 - val_accuracy: 0.9085 - val_loss: 0.2417 - learning_rate: 1.0000e-04
Epoch 3/10
541/543 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9451 - loss: 0.1647
Epoch 3: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.9451 - loss: 0.1647 - val_accuracy: 0.9074 - val_loss: 0.2421 - learning_rate: 5.0000e-05
Epoch 4/10
542/543 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9466 - loss: 0.1599
Epoch 4: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.94

In [ ]:
ann_probabilities = ann_model.predict(
    X_test_tfidf
)

ann_predictions = (
    ann_probabilities.ravel() >= 0.5
).astype(int)

ann_metrics = evaluate_model(
    y_test,
    ann_predictions,
    ann_probabilities.ravel()
)

ann_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.9033342296316214,
 'Precision': 0.9016524520255863,
 'Recall': 0.9062416287168498,
 'F1 Score': 0.9039412157648631,
 'ROC-AUC': 0.9669734103645027}

In [94]:
optimization_results.append({
    "Experiment": "E11",
    "Enhancement": "LR Scheduler",
    "Model": "ANN",
    'Accuracy': 0.9033342296316214,
    'Precision': 0.9016524520255863,
    'Recall': 0.9062416287168498,
    'F1 Score': 0.9039412157648631,
    'ROC-AUC': 0.9669734103645027
})

#  RNN


In [ ]:
rnn_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),

    SimpleRNN(
        64
    ),


    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - accuracy: 0.5052 - loss: 0.6974 - val_accuracy: 0.4952 - val_loss: 0.7131
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.5087 - loss: 0.6926 - val_accuracy: 0.5079 - val_loss: 0.6956
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.5473 - loss: 0.6628 - val_accuracy: 0.5092 - val_loss: 0.7121
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.5632 - loss: 0.6175 - val_accuracy: 0.5046 - val_loss: 0.7534
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 20ms/step - accuracy: 0.5759 - loss: 0.5956 - val_accuracy: 0.5026 - val_loss: 0.8437
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.5741 - loss: 0.5918 - val_accuracy: 0.5064 - val_loss: 0.8965
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5803 - loss: 0.5885 - val_accuracy: 0.4972 - val_loss: 0.9747
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5784 - loss: 0.5870 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


{'Accuracy': 0.49919333154073675,
 'Precision': 0.5005910165484634,
 'Recall': 0.9075810340208947,
 'F1 Score': 0.6452718788686792,
 'ROC-AUC': np.float64(0.5001797080638707)}

In [95]:
optimization_results.append({
    "Experiment": "E1",
    "Enhancement": "Dimension Size 256",
    "Model": "RNN",
    'Accuracy': 0.49919333154073675,
    'Precision': 0.5005910165484634,
    'Recall': 0.9075810340208947,
    'F1 Score': 0.6452718788686792,
    'ROC-AUC': 0.5001797080638707
})

In [ ]:
# Hidden Layers

rnn_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    SimpleRNN(
        128
    ),


    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 26ms/step - accuracy: 0.4971 - loss: 0.7056 - val_accuracy: 0.5052 - val_loss: 0.6948
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.5058 - loss: 0.6996 - val_accuracy: 0.4989 - val_loss: 0.6986
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.4980 - loss: 0.6987 - val_accuracy: 0.5097 - val_loss: 0.6987
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.5064 - loss: 0.6968 - val_accuracy: 0.4942 - val_loss: 0.6995
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.5134 - loss: 0.6938 - val_accuracy: 0.5040 - val_loss: 0.6955
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.5181 - loss: 0.6886 - val_accuracy: 0.5112 - val_loss: 0.6936
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.5363 - loss: 0.6775 - val_accuracy: 0.5194 - val_loss: 0.7012
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.5505 - loss: 0.6644 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.5021511158913686,
 'Precision': 0.5201612903225806,
 'Recall': 0.10366997053308331,
 'F1 Score': 0.17288362742908198,
 'ROC-AUC': np.float64(0.5255835812408063)}

In [96]:
optimization_results.append({
    "Experiment": "E2",
    "Enhancement": "Hidden Layer",
    "Model": "RNN",
    'Accuracy': 0.5021511158913686,
    'Precision': 0.5201612903225806,
    'Recall': 0.10366997053308331,
    'F1 Score': 0.17288362742908198,
    'ROC-AUC': 0.5255835812408063
})

In [ ]:
# Dropout

rnn_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    SimpleRNN(
        128
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 20s 26ms/step - accuracy: 0.4983 - loss: 0.7093 - val_accuracy: 0.5018 - val_loss: 0.6932
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5022 - loss: 0.6940 - val_accuracy: 0.5017 - val_loss: 0.6932
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.4979 - loss: 0.6936 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.5003 - loss: 0.6933 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.4991 - loss: 0.6933 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.5021 - loss: 0.6932 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.4977 - loss: 0.6934 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.4995 - loss: 0.6934 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step


{'Accuracy': 0.5018822264049476,
 'Precision': 0.5018822264049476,
 'Recall': 1.0,
 'F1 Score': 0.6683376600125325,
 'ROC-AUC': np.float64(0.5)}

In [97]:
optimization_results.append({
    "Experiment": "E3",
    "Enhancement": "DropOut 0.5",
    "Model": "RNN",
    'Accuracy': 0.5018822264049476,
    'Precision': 0.5018822264049476,
    'Recall': 1.0,
    'F1 Score': 0.6683376600125325,
    'ROC-AUC': 0.50000000000000
})

In [ ]:
# Dropout

rnn_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    SimpleRNN(
        128
    ),

    tf.keras.layers.Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 25s 31ms/step - accuracy: 0.4982 - loss: 0.7149 - val_accuracy: 0.4936 - val_loss: 0.6941
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5023 - loss: 0.6979 - val_accuracy: 0.5083 - val_loss: 0.6938
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.4985 - loss: 0.6953 - val_accuracy: 0.5063 - val_loss: 0.6964
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5016 - loss: 0.6949 - val_accuracy: 0.5044 - val_loss: 0.6933
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5009 - loss: 0.6940 - val_accuracy: 0.5084 - val_loss: 0.6929
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.5000 - loss: 0.6936 - val_accuracy: 0.5068 - val_loss: 0.6930
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5019 - loss: 0.6933 - val_accuracy: 0.5112 - val_loss: 0.6930
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.5027 - loss: 0.6937 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step


{'Accuracy': 0.5001344447432106,
 'Precision': 0.5018315018315018,
 'Recall': 0.5504955799624967,
 'F1 Score': 0.525038323965253,
 'ROC-AUC': np.float64(0.5033711439678137)}

In [98]:
optimization_results.append({
    "Experiment": "E4",
    "Enhancement": "DropOut 0.3",
    "Model": "RNN",
    'Accuracy': 0.5001344447432106,
    'Precision': 0.5018315018315018,
    'Recall': 0.5504955799624967,
    'F1 Score': 0.525038323965253,
    'ROC-AUC': 0.5033711439678137
})

In [ ]:
# BatchNormalization

rnn_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    SimpleRNN(
        128
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 23s 30ms/step - accuracy: 0.4999 - loss: 0.8069 - val_accuracy: 0.5006 - val_loss: 0.6960
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.5035 - loss: 0.7013 - val_accuracy: 0.4908 - val_loss: 0.6947
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.5019 - loss: 0.6949 - val_accuracy: 0.5040 - val_loss: 0.6935
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5000 - loss: 0.6946 - val_accuracy: 0.5073 - val_loss: 0.6929
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.5049 - loss: 0.6942 - val_accuracy: 0.5026 - val_loss: 0.6936
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.5027 - loss: 0.6940 - val_accuracy: 0.4975 - val_loss: 0.6929
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.5215 - loss: 0.6921 - val_accuracy: 0.5279 - val_loss: 0.6919
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.5317 - loss: 0.6907 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.5172089271309491,
 'Precision': 0.5145670906852687,
 'Recall': 0.671845700508974,
 'F1 Score': 0.5827814569536424,
 'ROC-AUC': np.float64(0.5281140269536789)}

In [99]:
optimization_results.append({
    "Experiment": "E5",
    "Enhancement": "Batchnormalization",
    "Model": "RNN",
    'Accuracy': 0.5172089271309491,
    'Precision': 0.5145670906852687,
    'Recall': 0.671845700508974,
    'F1 Score': 0.5827814569536424,
    'ROC-AUC': 0.5281140269536789
})

In [ ]:
# Recurrent dropout

rnn_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    SimpleRNN(
        128,
        recurrent_dropout=0.2
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.5),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 27s 33ms/step - accuracy: 0.5013 - loss: 0.8066 - val_accuracy: 0.4940 - val_loss: 0.6946
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5003 - loss: 0.7017 - val_accuracy: 0.5054 - val_loss: 0.6932
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.4989 - loss: 0.6949 - val_accuracy: 0.4971 - val_loss: 0.6945
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5030 - loss: 0.6942 - val_accuracy: 0.5024 - val_loss: 0.6932
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.4993 - loss: 0.6944 - val_accuracy: 0.4991 - val_loss: 0.6934
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.4997 - loss: 0.6944 - val_accuracy: 0.4964 - val_loss: 0.6955
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.5051 - loss: 0.6939 - val_accuracy: 0.4981 - val_loss: 0.6937
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - accuracy: 0.4946 - loss: 0.6946 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.5002688894864211,
 'Precision': 0.5266666666666666,
 'Recall': 0.04232520760782213,
 'F1 Score': 0.07835358294073891,
 'ROC-AUC': np.float64(0.4986405668811523)}

In [100]:
optimization_results.append({
    "Experiment": "E6",
    "Enhancement": "Recurrent Dropout",
    "Model": "RNN",
    'Accuracy': 0.5002688894864211,
    'Precision': 0.5266666666666666,
    'Recall': 0.04232520760782213,
    'F1 Score': 0.07835358294073891,
    'ROC-AUC': 0.4986405668811523
})

In [ ]:
# Different Optimizers

rnn_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    SimpleRNN(
        128,
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
rnn_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 35s 55ms/step - accuracy: 0.5072 - loss: 0.6996 - val_accuracy: 0.5038 - val_loss: 0.6963
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 27s 50ms/step - accuracy: 0.5053 - loss: 0.6946 - val_accuracy: 0.5228 - val_loss: 0.6918
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.4957 - loss: 0.6945 - val_accuracy: 0.4947 - val_loss: 0.6938
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.4998 - loss: 0.6938 - val_accuracy: 0.4931 - val_loss: 0.6937
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.5025 - loss: 0.6938 - val_accuracy: 0.4975 - val_loss: 0.6936
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.4981 - loss: 0.6939 - val_accuracy: 0.4919 - val_loss: 0.6933
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.5085 - loss: 0.6934 - val_accuracy: 0.4983 - val_loss: 0.6935
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.5026 - loss: 0.6935 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.5063189029308954,
 'Precision': 0.5131182795698924,
 'Recall': 0.31958210554513794,
 'F1 Score': 0.39385935952459555,
 'ROC-AUC': np.float64(0.5053490533603889)}

In [101]:
optimization_results.append({
    "Experiment": "E7",
    "Enhancement": "New Optimizers Rmsprop",
    "Model": "RNN",
    'Accuracy': 0.5063189029308954,
    'Precision': 0.5131182795698924,
    'Recall': 0.31958210554513794,
    'F1 Score': 0.39385935952459555,
    'ROC-AUC': 0.5053490533603889
})

In [ ]:
# Learning Rate

rnn_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 28s 40ms/step - accuracy: 0.5037 - loss: 0.6931 - val_accuracy: 0.4912 - val_loss: 0.6933
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 28s 24ms/step - accuracy: 0.5029 - loss: 0.6932 - val_accuracy: 0.4878 - val_loss: 0.6933
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5047 - loss: 0.6933 - val_accuracy: 0.4884 - val_loss: 0.6932
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.5060 - loss: 0.6929 - val_accuracy: 0.4855 - val_loss: 0.6935
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.5015 - loss: 0.6933 - val_accuracy: 0.4861 - val_loss: 0.6934
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.5057 - loss: 0.6931 - val_accuracy: 0.4884 - val_loss: 0.6933
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.5061 - loss: 0.6931 - val_accuracy: 0.4851 - val_loss: 0.6934
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.5055 - loss: 0.6931 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


{'Accuracy': 0.5060500134444743,
 'Precision': 0.5120951209512095,
 'Recall': 0.334583444950442,
 'F1 Score': 0.40473104342190536,
 'ROC-AUC': np.float64(0.5097830814130673)}

In [102]:
optimization_results.append({
    "Experiment": "E8",
    "Enhancement": "Learning rate rmsprop 0.0001",
    "Model": "RNN",
    'Accuracy': 0.5060500134444743,
    'Precision': 0.5120951209512095,
    'Recall': 0.334583444950442,
    'F1 Score': 0.40473104342190536,
    'ROC-AUC': 0.5097830814130673
})

In [ ]:
# Learning Rate

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 26ms/step - accuracy: 0.5023 - loss: 0.6931 - val_accuracy: 0.4899 - val_loss: 0.6933
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.5051 - loss: 0.6931 - val_accuracy: 0.4795 - val_loss: 0.6935
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.5050 - loss: 0.6931 - val_accuracy: 0.4911 - val_loss: 0.6934
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5058 - loss: 0.6931 - val_accuracy: 0.4904 - val_loss: 0.6932
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5084 - loss: 0.6929 - val_accuracy: 0.5112 - val_loss: 0.6930
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.5038 - loss: 0.6927 - val_accuracy: 0.5080 - val_loss: 0.6931
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - accuracy: 0.4992 - loss: 0.6935 - val_accuracy: 0.4974 - val_loss: 0.6931
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 20ms/step - accuracy: 0.4985 - loss: 0.6934 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.5037644528098951,
 'Precision': 0.5724137931034483,
 'Recall': 0.04446825609429413,
 'F1 Score': 0.08252547849863286,
 'ROC-AUC': np.float64(0.5056127770228184)}

In [103]:
optimization_results.append({
    "Experiment": "E8",
    "Enhancement": "Learning rate adam 0.0001",
    "Model": "RNN",
    'Accuracy': 0.5037644528098951,
    'Precision': 0.5724137931034483,
    'Recall': 0.04446825609429413,
    'F1 Score': 0.08252547849863286,
    'ROC-AUC': 0.5056127770228184
})

In [ ]:
# Learning Rate

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 25ms/step - accuracy: 0.5060 - loss: 0.6926 - val_accuracy: 0.4987 - val_loss: 0.6933
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5090 - loss: 0.6910 - val_accuracy: 0.5083 - val_loss: 0.6918
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.5072 - loss: 0.6927 - val_accuracy: 0.5024 - val_loss: 0.6929
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.5043 - loss: 0.6926 - val_accuracy: 0.5291 - val_loss: 0.6920
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.5322 - loss: 0.6906 - val_accuracy: 0.5456 - val_loss: 0.6883
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.5606 - loss: 0.6786 - val_accuracy: 0.5606 - val_loss: 0.6841
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.5653 - loss: 0.6701 - val_accuracy: 0.5134 - val_loss: 0.6919
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.5370 - loss: 0.6738 - 

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.5232589405754235,
 'Precision': 0.673469387755102,
 'Recall': 0.0972408250736673,
 'F1 Score': 0.1699438202247191,
 'ROC-AUC': np.float64(0.538737770470397)}

In [104]:
optimization_results.append({
    "Experiment": "E9",
    "Enhancement": "Learning rate adam 0.0005",
    "Model": "RNN",
    'Accuracy': 0.5232589405754235,
    'Precision': 0.673469387755102,
    'Recall': 0.0972408250736673,
    'F1 Score': 0.1699438202247191,
    'ROC-AUC': 0.538737770470397
})

In [ ]:
# Batchsize Tuning

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=32
)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 21ms/step - accuracy: 0.5534 - loss: 0.6450 - val_accuracy: 0.5179 - val_loss: 0.7329
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 25s 23ms/step - accuracy: 0.5624 - loss: 0.6273 - val_accuracy: 0.5188 - val_loss: 0.7571
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.5642 - loss: 0.6179 - val_accuracy: 0.5206 - val_loss: 0.7906
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 26s 24ms/step - accuracy: 0.5705 - loss: 0.6105 - val_accuracy: 0.5192 - val_loss: 0.8042
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 19ms/step - accuracy: 0.5697 - loss: 0.6071 - val_accuracy: 0.5177 - val_loss: 0.8598
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 23s 21ms/step - accuracy: 0.5718 - loss: 0.6100 - val_accuracy: 0.5108 - val_loss: 0.8225
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 21s 20ms/step - accuracy: 0.5694 - loss: 0.6107 - val_accuracy: 0.5225 - val_loss: 0.8159
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.5692 -

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.5212422694272654,
 'Precision': 0.6563636363636364,
 'Recall': 0.0967050629520493,
 'F1 Score': 0.16857342983889798,
 'ROC-AUC': np.float64(0.5348182837319556)}

In [109]:
optimization_results.append({
    "Experiment": "E10",
    "Enhancement": "BatchSize 32",
    "Model": "RNN",
    'Accuracy': 0.5212422694272654,
    'Precision': 0.6563636363636364,
    'Recall': 0.0967050629520493,
    'F1 Score': 0.16857342983889798,
    'ROC-AUC': 0.5348182837319556
})

In [ ]:
# Early stopping

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
# Early stopping

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=20,
    batch_size=64,
    callbacks=[early_stopping]

)

Epoch 1/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 23s 34ms/step - accuracy: 0.5710 - loss: 0.6025 - val_accuracy: 0.5201 - val_loss: 0.8831
Epoch 2/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 38s 32ms/step - accuracy: 0.5725 - loss: 0.5996 - val_accuracy: 0.5204 - val_loss: 0.9388
Epoch 3/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.5758 - loss: 0.5978 - val_accuracy: 0.5222 - val_loss: 0.9412


In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.5228556063457919,
 'Precision': 0.6832669322709163,
 'Recall': 0.09188320385748727,
 'F1 Score': 0.16198347107438016,
 'ROC-AUC': np.float64(0.5369703700409919)}

In [110]:
optimization_results.append({
    "Experiment": "E11",
    "Enhancement": "EarlyStopping",
    "Model": "RNN",
    'Accuracy': 0.5228556063457919,
    'Precision': 0.6832669322709163,
    'Recall': 0.09188320385748727,
    'F1 Score': 0.16198347107438016,
    'ROC-AUC': 0.5369703700409919
})

In [ ]:
# Learning rate Scheduler

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
# Learning rate Scheduler

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_history = rnn_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64,
    callbacks=[lr_scheduler]

)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 22s 27ms/step - accuracy: 0.5724 - loss: 0.6017 - val_accuracy: 0.5194 - val_loss: 0.9198 - learning_rate: 5.0000e-04
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5816 - loss: 0.5966
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.5748 - loss: 0.5992 - val_accuracy: 0.5213 - val_loss: 0.9253 - learning_rate: 5.0000e-04
Epoch 3/10
541/543 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5696 - loss: 0.5966
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.5733 - loss: 0.5962 - val_accuracy: 0.5221 - val_loss: 0.9485 - learning_rate: 2.5000e-04
Epoch 4/10
541/543 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5747 - loss: 0.5968
Epoch 4: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.575

In [ ]:
rnn_probabilities = rnn_model.predict(
    X_test_padded
)

rnn_predictions = (
    rnn_probabilities.ravel() >= 0.5
).astype(int)

rnn_metrics = evaluate_model(
    y_test,
    rnn_predictions,
    rnn_probabilities.ravel()
)

rnn_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step


{'Accuracy': 0.521645603656897,
 'Precision': 0.6576576576576577,
 'Recall': 0.0977765871952853,
 'F1 Score': 0.17024253731343283,
 'ROC-AUC': np.float64(0.5340844848423062)}

In [111]:
optimization_results.append({
    "Experiment": "E12",
    "Enhancement": "lr scheduler",
    "Model": "RNN",
    'Accuracy': 0.521645603656897,
    'Precision': 0.6576576576576577,
    'Recall': 0.0977765871952853,
    'F1 Score': 0.17024253731343283,
    'ROC-AUC': 0.5340844848423062
})

# LSTM

In [ ]:
# LSTM

lstm_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),

    LSTM(
        64
    ),


    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 17ms/step - accuracy: 0.5246 - loss: 0.6832 - val_accuracy: 0.5530 - val_loss: 0.6708
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.6439 - loss: 0.5731 - val_accuracy: 0.8550 - val_loss: 0.3532
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.8980 - loss: 0.2622 - val_accuracy: 0.8836 - val_loss: 0.2916
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.9535 - loss: 0.1349 - val_accuracy: 0.8711 - val_loss: 0.3571
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9798 - loss: 0.0675 - val_accuracy: 0.8657 - val_loss: 0.4662
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9890 - loss: 0.0404 - val_accuracy: 0.8634 - val_loss: 0.5069
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9921 - loss: 0.0292 - val_accuracy: 0.8707 - val_loss: 0.5421
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9942 - loss: 0.0222 - val_acc

In [ ]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


{'Accuracy': 0.8628663619252487,
 'Precision': 0.8750345590268178,
 'Recall': 0.8478435574604876,
 'F1 Score': 0.8612244897959184,
 'ROC-AUC': np.float64(0.9304515332304468)}

In [112]:
optimization_results.append({
    "Experiment": "E1",
    "Enhancement": "Embedding Dimension",
    "Model": "LSTM",
    'Accuracy': 0.8628663619252487,
    'Precision': 0.8750345590268178,
    'Recall': 0.8478435574604876,
    'F1 Score': 0.8612244897959184,
    'ROC-AUC': 0.9304515332304468
})

In [ ]:
# LSTM

lstm_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    LSTM(128),


    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.5185 - loss: 0.6899 - val_accuracy: 0.5171 - val_loss: 0.6852
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.5563 - loss: 0.6550 - val_accuracy: 0.7638 - val_loss: 0.5440
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.7447 - loss: 0.5246 - val_accuracy: 0.8431 - val_loss: 0.3832
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.8916 - loss: 0.2760 - val_accuracy: 0.8810 - val_loss: 0.2887
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.9401 - loss: 0.1680 - val_accuracy: 0.8843 - val_loss: 0.3105
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9650 - loss: 0.1097 - val_accuracy: 0.8829 - val_loss: 0.3467
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9822 - loss: 0.0652 - val_accuracy: 0.8773 - val_loss: 0.4257
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - accuracy: 0.9910 - loss: 0.0393 - val

In [ ]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8683785963968809,
 'Precision': 0.8578482328482329,
 'Recall': 0.8842753817305117,
 'F1 Score': 0.8708613639361562,
 'ROC-AUC': np.float64(0.9306577401900762)}

In [113]:
optimization_results.append({
    "Experiment": "E2",
    "Enhancement": "Hidden layer 128",
    "Model": "LSTM",
    'Accuracy': 0.8683785963968809,
    'Precision': 0.8578482328482329,
    'Recall': 0.8842753817305117,
    'F1 Score': 0.8708613639361562,
    'ROC-AUC': 0.9306577401900762
})

In [ ]:
# Drop out


lstm_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    LSTM(128),

    tf.keras.layers.Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.9927 - loss: 0.0268 - val_accuracy: 0.8782 - val_loss: 0.6637
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9963 - loss: 0.0152 - val_accuracy: 0.8661 - val_loss: 0.7276
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.9982 - loss: 0.0088 - val_accuracy: 0.8738 - val_loss: 0.8851
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9976 - loss: 0.0094 - val_accuracy: 0.8714 - val_loss: 0.7476
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9967 - loss: 0.0131 - val_accuracy: 0.8689 - val_loss: 0.7544
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9976 - loss: 0.0083 - val_accuracy: 0.8658 - val_loss: 1.0032
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9991 - loss: 0.0035 - val_accuracy: 0.8681 - val_loss: 0.9838
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9986 - loss: 0.0052 - val_ac

In [ ]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


{'Accuracy': 0.860446356547459,
 'Precision': 0.8644847173383825,
 'Recall': 0.8561478703455666,
 'F1 Score': 0.8602960969044414,
 'ROC-AUC': np.float64(0.9315783689477769)}

In [114]:
optimization_results.append({
    "Experiment": "E3",
    "Enhancement": "DropOut 0.3",
    "Model": "LSTM",
    'Accuracy': 0.860446356547459,
    'Precision': 0.8644847173383825,
    'Recall': 0.8561478703455666,
    'F1 Score': 0.8602960969044414,
    'ROC-AUC': 0.9315783689477769
})

In [ ]:
# Drop out


lstm_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    LSTM(128),

    tf.keras.layers.Dropout(0.5),

    Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.5056 - loss: 0.6931 - val_accuracy: 0.5083 - val_loss: 0.6917
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.5393 - loss: 0.6740 - val_accuracy: 0.5306 - val_loss: 0.6906
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5556 - loss: 0.6332 - val_accuracy: 0.5343 - val_loss: 0.6926
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.7466 - loss: 0.4797 - val_accuracy: 0.8532 - val_loss: 0.3614
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9114 - loss: 0.2488 - val_accuracy: 0.8767 - val_loss: 0.3078
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9548 - loss: 0.1414 - val_accuracy: 0.8731 - val_loss: 0.3640
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9766 - loss: 0.0831 - val_accuracy: 0.8704 - val_loss: 0.4455
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9863 - loss: 0.0498 - val_acc

In [ ]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


{'Accuracy': 0.8507663350363001,
 'Precision': 0.88314344142565,
 'Recall': 0.8098044468256095,
 'F1 Score': 0.844885410844047,
 'ROC-AUC': np.float64(0.9288287379620723)}

In [115]:
optimization_results.append({
    "Experiment": "E4",
    "Enhancement": "DropOut 0.5",
    "Model": "LSTM",
    'Accuracy': 0.8507663350363001,
    'Precision': 0.88314344142565,
    'Recall': 0.8098044468256095,
    'F1 Score': 0.844885410844047,
    'ROC-AUC': 0.9288287379620723
})

In [12]:
# Recurrent Dropout

lstm_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    LSTM(
        64,
        recurrent_dropout=0.2
    ),

    tf.keras.layers.Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [13]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 342s 616ms/step - accuracy: 0.5151 - loss: 0.6910 - val_accuracy: 0.7169 - val_loss: 0.6070
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 342s 630ms/step - accuracy: 0.5308 - loss: 0.6794 - val_accuracy: 0.5292 - val_loss: 0.6825
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 379s 625ms/step - accuracy: 0.5631 - loss: 0.6376 - val_accuracy: 0.7719 - val_loss: 0.5753
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 338s 623ms/step - accuracy: 0.8381 - loss: 0.4082 - val_accuracy: 0.8695 - val_loss: 0.3343
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 342s 630ms/step - accuracy: 0.9144 - loss: 0.2402 - val_accuracy: 0.8786 - val_loss: 0.3154
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 377s 621ms/step - accuracy: 0.9483 - loss: 0.1586 - val_accuracy: 0.8697 - val_loss: 0.3654
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 340s 625ms/step - accuracy: 0.9695 - loss: 0.1017 - val_accuracy: 0.8751 - val_loss: 0.4352
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 337s 621ms/step - accuracy: 0.9827 -

In [14]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 37s 156ms/step


{'Accuracy': 0.867437483194407,
 'Precision': 0.8510094556606185,
 'Recall': 0.8920439324939726,
 'F1 Score': 0.8710436829714884,
 'ROC-AUC': np.float64(0.9332521013841244)}

In [116]:
optimization_results.append({
    "Experiment": "E5",
    "Enhancement": "recurrent dropout",
    "Model": "LSTM",
    'Accuracy': 0.867437483194407,
    'Precision': 0.8510094556606185,
    'Recall': 0.8920439324939726,
    'F1 Score': 0.8710436829714884,
    'ROC-AUC': 0.9332521013841244
})

In [15]:
# Recurrent Dropout

lstm_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    LSTM(64),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [16]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.5162 - loss: 0.7421 - val_accuracy: 0.5041 - val_loss: 0.7345
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.6990 - loss: 0.5231 - val_accuracy: 0.8552 - val_loss: 0.3420
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9121 - loss: 0.2288 - val_accuracy: 0.8353 - val_loss: 0.4048
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9518 - loss: 0.1352 - val_accuracy: 0.8753 - val_loss: 0.3642
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9723 - loss: 0.0827 - val_accuracy: 0.8681 - val_loss: 0.4173
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9815 - loss: 0.0565 - val_accuracy: 0.8086 - val_loss: 0.6745
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9872 - loss: 0.0402 - val_accuracy: 0.8580 - val_loss: 0.5190
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9886 - loss: 0.0352 - 

In [17]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8640763646141436,
 'Precision': 0.866254036598493,
 'Recall': 0.8623091347441736,
 'F1 Score': 0.8642770841723721,
 'ROC-AUC': np.float64(0.9337617984254667)}

In [18]:
# BatchNormalization

lstm_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    LSTM(64),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [19]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.5078 - loss: 0.7500 - val_accuracy: 0.5096 - val_loss: 0.6917
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 27s 30ms/step - accuracy: 0.6175 - loss: 0.6079 - val_accuracy: 0.8450 - val_loss: 0.3823
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 25ms/step - accuracy: 0.8962 - loss: 0.2627 - val_accuracy: 0.8219 - val_loss: 0.4322
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.9464 - loss: 0.1506 - val_accuracy: 0.8769 - val_loss: 0.3246
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9692 - loss: 0.0919 - val_accuracy: 0.8266 - val_loss: 0.5440
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - accuracy: 0.9820 - loss: 0.0567 - val_accuracy: 0.8622 - val_loss: 0.5753
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9882 - loss: 0.0364 - val_accuracy: 0.8699 - val_loss: 0.5577
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 25s 37ms/step - accuracy: 0.9911 - loss: 0.0286 - 

In [20]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


{'Accuracy': 0.8471363269696155,
 'Precision': 0.9173633440514469,
 'Recall': 0.7642646664880793,
 'F1 Score': 0.8338448049101271,
 'ROC-AUC': np.float64(0.9326183692659084)}

In [117]:
optimization_results.append({
    "Experiment": "E6",
    "Enhancement": "Batchnormalization",
    "Model": "LSTM",
    'Accuracy': 0.8471363269696155,
    'Precision': 0.9173633440514469,
    'Recall': 0.7642646664880793,
    'F1 Score': 0.8338448049101271,
    'ROC-AUC': 0.9326183692659084
})

In [22]:
# New OPtimizers


lstm_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),

    LSTM(64),

    tf.keras.layers.Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [23]:
lstm_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.5018 - loss: 0.6939 - val_accuracy: 0.5101 - val_loss: 0.6921
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.5110 - loss: 0.6931 - val_accuracy: 0.5288 - val_loss: 0.6807
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.5347 - loss: 0.6772 - val_accuracy: 0.5372 - val_loss: 0.6778
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.5503 - loss: 0.6596 - val_accuracy: 0.5442 - val_loss: 0.6610
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.7262 - loss: 0.5524 - val_accuracy: 0.8218 - val_loss: 0.4630
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.8262 - loss: 0.4621 - val_accuracy: 0.8227 - val_loss: 0.4627
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8417 - loss: 0.4360 - val_accuracy: 0.8281 - val_loss: 0.4562
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.8546 - loss: 0.3877 - val_acc

In [24]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8740252756117236,
 'Precision': 0.8322243346007605,
 'Recall': 0.9381194749531209,
 'F1 Score': 0.882004785291525,
 'ROC-AUC': np.float64(0.9335335753300702)}

In [118]:
optimization_results.append({
    "Experiment": "E7",
    "Enhancement": "New Optimizers rmsprop",
    "Model": "LSTM",
    'Accuracy': 0.8740252756117236,
    'Precision': 0.8322243346007605,
    'Recall': 0.9381194749531209,
    'F1 Score': 0.882004785291525,
    'ROC-AUC': 0.9335335753300702
})

In [25]:
# Learning Rate

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.9449 - loss: 0.1645 - val_accuracy: 0.8950 - val_loss: 0.3019
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9578 - loss: 0.1338 - val_accuracy: 0.8900 - val_loss: 0.3090
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9678 - loss: 0.1090 - val_accuracy: 0.8919 - val_loss: 0.3155
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9777 - loss: 0.0854 - val_accuracy: 0.8879 - val_loss: 0.3676
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9849 - loss: 0.0649 - val_accuracy: 0.8869 - val_loss: 0.3958
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 17ms/step - accuracy: 0.9883 - loss: 0.0536 - val_accuracy: 0.8849 - val_loss: 0.4527
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9904 - loss: 0.0463 - val_accuracy: 0.8817 - val_loss: 0.4288
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9926 - loss: 0.0385 - val_

In [26]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8767141704759344,
 'Precision': 0.8832335329341318,
 'Recall': 0.8692740423252077,
 'F1 Score': 0.8761981909004996,
 'ROC-AUC': np.float64(0.944330302770671)}

In [119]:
optimization_results.append({
    "Experiment": "E8",
    "Enhancement": "learning rate adam 0.0001",
    "Model": "LSTM",
    'Accuracy': 0.8767141704759344,
    'Precision': 0.8832335329341318,
    'Recall': 0.8692740423252077,
    'F1 Score': 0.8761981909004996,
    'ROC-AUC': 0.944330302770671
})

In [27]:
# Learning Rate

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.9767 - loss: 0.0736 - val_accuracy: 0.8840 - val_loss: 0.4046
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9874 - loss: 0.0443 - val_accuracy: 0.8825 - val_loss: 0.4822
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.9910 - loss: 0.0338 - val_accuracy: 0.8826 - val_loss: 0.5975
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9918 - loss: 0.0301 - val_accuracy: 0.8812 - val_loss: 0.6466
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.9946 - loss: 0.0203 - val_accuracy: 0.8766 - val_loss: 0.5629
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9952 - loss: 0.0193 - val_accuracy: 0.8789 - val_loss: 0.6510
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9951 - loss: 0.0173 - val_accuracy: 0.8761 - val_loss: 0.6760
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9941 - loss: 0.0203 - val_a

In [28]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step


{'Accuracy': 0.8695885990857758,
 'Precision': 0.8638398735844087,
 'Recall': 0.8786498794535227,
 'F1 Score': 0.8711819389110226,
 'ROC-AUC': np.float64(0.9395747812937317)}

In [120]:
optimization_results.append({
    "Experiment": "E9",
    "Enhancement": "learning rate adam 0.0005",
    "Model": "LSTM",
    'Accuracy': 0.8695885990857758,
    'Precision': 0.8638398735844087,
    'Recall': 0.8786498794535227,
    'F1 Score': 0.8711819389110226,
    'ROC-AUC': 0.9395747812937317
})

In [29]:
# Learning Rate

lstm_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.9997 - loss: 0.0018 - val_accuracy: 0.8766 - val_loss: 1.0345
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9999 - loss: 0.0013 - val_accuracy: 0.8744 - val_loss: 1.1076
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9999 - loss: 0.0014 - val_accuracy: 0.8767 - val_loss: 1.1905
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9999 - loss: 0.0013 - val_accuracy: 0.8769 - val_loss: 1.2397
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9999 - loss: 9.4110e-04 - val_accuracy: 0.8763 - val_loss: 1.2604
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9999 - loss: 9.5169e-04 - val_accuracy: 0.8763 - val_loss: 1.2985
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9999 - loss: 8.7067e-04 - val_accuracy: 0.8767 - val_loss: 1.3128
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9999 - loss: 8.27

In [30]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8713363807475127,
 'Precision': 0.8759479956663055,
 'Recall': 0.8663273506563086,
 'F1 Score': 0.8711111111111111,
 'ROC-AUC': np.float64(0.9358019964911557)}

In [121]:
optimization_results.append({
    "Experiment": "E10",
    "Enhancement": "learning rate rmsprop 0.0001",
    "Model": "LSTM",
    'Accuracy': 0.8713363807475127,
    'Precision': 0.8759479956663055,
    'Recall': 0.8663273506563086,
    'F1 Score': 0.8711111111111111,
    'ROC-AUC': 0.9358019964911557
})

In [31]:
# Batch Size

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=32
)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 28s 23ms/step - accuracy: 0.9996 - loss: 0.0024 - val_accuracy: 0.8774 - val_loss: 1.4066
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.9998 - loss: 9.4464e-04 - val_accuracy: 0.8736 - val_loss: 1.2924
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - accuracy: 1.0000 - loss: 2.3425e-04 - val_accuracy: 0.8755 - val_loss: 1.3309
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 48s 25ms/step - accuracy: 1.0000 - loss: 3.9533e-05 - val_accuracy: 0.8763 - val_loss: 1.4055
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 24s 22ms/step - accuracy: 1.0000 - loss: 3.1568e-05 - val_accuracy: 0.8758 - val_loss: 1.4959
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 24s 22ms/step - accuracy: 1.0000 - loss: 2.8118e-05 - val_accuracy: 0.8761 - val_loss: 1.6038
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 1.0000 - loss: 1.9356e-05 - val_accuracy: 0.8758 - val_loss: 1.7340
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 17s 16ms/s

In [33]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8732186071524604,
 'Precision': 0.8739946380697051,
 'Recall': 0.8732922582373426,
 'F1 Score': 0.8736433069811068,
 'ROC-AUC': np.float64(0.9207541665265804)}

In [122]:
optimization_results.append({
    "Experiment": "E11",
    "Enhancement": "Batch size 32",
    "Model": "LSTM",
    'Accuracy': 0.8732186071524604,
    'Precision': 0.8739946380697051,
    'Recall': 0.8732922582373426,
    'F1 Score': 0.8736433069811068,
    'ROC-AUC': 0.9207541665265804
})

In [34]:
# Batch Size

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=16
)

Epoch 1/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 35s 15ms/step - accuracy: 1.0000 - loss: 6.7550e-04 - val_accuracy: 0.8730 - val_loss: 2.2647
Epoch 2/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 34s 15ms/step - accuracy: 1.0000 - loss: 2.8181e-06 - val_accuracy: 0.8761 - val_loss: 2.3599
Epoch 3/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - accuracy: 0.9996 - loss: 0.0043 - val_accuracy: 0.8754 - val_loss: 1.5089
Epoch 4/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - accuracy: 1.0000 - loss: 7.8579e-05 - val_accuracy: 0.8761 - val_loss: 1.5819
Epoch 5/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 33s 15ms/step - accuracy: 1.0000 - loss: 1.9110e-05 - val_accuracy: 0.8750 - val_loss: 1.6955
Epoch 6/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 32s 15ms/step - accuracy: 1.0000 - loss: 1.1261e-05 - val_accuracy: 0.8748 - val_loss: 1.8451
Epoch 7/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 33s 15ms/step - accuracy: 1.0000 - loss: 4.4868e-06 - val_accuracy: 0.8770 - val_loss: 2.0078
Epoch 8/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 41s 19ms/s

In [35]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8702608228018285,
 'Precision': 0.8754747693977211,
 'Recall': 0.8644521832306455,
 'F1 Score': 0.8699285618007818,
 'ROC-AUC': np.float64(0.9269572579680155)}

In [123]:
optimization_results.append({
    "Experiment": "E12",
    "Enhancement": "Batch size 16",
    "Model": "LSTM",
    'Accuracy': 0.8702608228018285,
    'Precision': 0.8754747693977211,
    'Recall': 0.8644521832306455,
    'F1 Score': 0.8699285618007818,
    'ROC-AUC': 0.9269572579680155
})

In [36]:
# Early Stopping

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor = "val_loss",
    patience = 5,
    restore_best_weights = True
)


In [41]:
# Learning Rate

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64,
    callbacks=[early_stopping]
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 1.0000 - loss: 1.2546e-05 - val_accuracy: 0.8759 - val_loss: 1.8623
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 1.0000 - loss: 3.4070e-06 - val_accuracy: 0.8763 - val_loss: 1.9905
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 1.0000 - loss: 4.2219e-06 - val_accuracy: 0.8777 - val_loss: 2.0761
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 1.0000 - loss: 1.4848e-06 - val_accuracy: 0.8769 - val_loss: 2.1532
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 1.0000 - loss: 1.7368e-05 - val_accuracy: 0.8767 - val_loss: 2.2248


In [39]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8699919333154074,
 'Precision': 0.8752034725990233,
 'Recall': 0.8641843021698365,
 'F1 Score': 0.8696589836905243,
 'ROC-AUC': np.float64(0.9356759730933176)}

In [124]:
optimization_results.append({
    "Experiment": "E13",
    "Enhancement": "Early stopping",
    "Model": "LSTM",
    'Accuracy': 0.8699919333154074,
    'Precision': 0.8752034725990233,
    'Recall': 0.8641843021698365,
    'F1 Score': 0.8696589836905243,
    'ROC-AUC': 0.9356759730933176
})

In [40]:
# LR Schedule

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

In [42]:
# Learning Rate

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_history = lstm_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64,
    callbacks=[lr_scheduler]
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 1.0000 - loss: 6.0265e-06 - val_accuracy: 0.8781 - val_loss: 2.1609 - learning_rate: 1.0000e-04
Epoch 2/10
540/543 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 1.0607e-06
Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 1.0000 - loss: 1.2845e-06 - val_accuracy: 0.8792 - val_loss: 2.2737 - learning_rate: 1.0000e-04
Epoch 3/10
541/543 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 7.0900e-07
Epoch 3: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 1.0000 - loss: 1.1466e-06 - val_accuracy: 0.8790 - val_loss: 2.3333 - learning_rate: 5.0000e-05
Epoch 4/10
540/543 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 1.4588e-06
Epoch 4: ReduceLROnPlateau reducing learning rate to 1.249999968422344e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms

In [43]:
lstm_probabilities = lstm_model.predict(
    X_test_padded
)

lstm_predictions = (
    lstm_probabilities.ravel() >= 0.5
).astype(int)

lstm_metrics = evaluate_model(
    y_test,
    lstm_predictions,
    lstm_probabilities.ravel()
)

lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step


{'Accuracy': 0.8726808281796182,
 'Precision': 0.8736587982832618,
 'Recall': 0.8724886150549156,
 'F1 Score': 0.8730733145690927,
 'ROC-AUC': np.float64(0.9234218786885613)}

In [125]:
optimization_results.append({
    "Experiment": "E14",
    "Enhancement": "LR Schedule",
    "Model": "LSTM",
    'Accuracy': 0.8726808281796182,
    'Precision': 0.8736587982832618,
    'Recall': 0.8724886150549156,
    'F1 Score': 0.8730733145690927,
    'ROC-AUC': 0.9234218786885613
})

# GRU

In [44]:
# GRU

gru_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),

    GRU(
        64
    ),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [45]:
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [46]:
gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - accuracy: 0.5076 - loss: 0.6921 - val_accuracy: 0.5228 - val_loss: 0.6863
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.7092 - loss: 0.5047 - val_accuracy: 0.8785 - val_loss: 0.2915
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9263 - loss: 0.1949 - val_accuracy: 0.8836 - val_loss: 0.2889
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9744 - loss: 0.0813 - val_accuracy: 0.8750 - val_loss: 0.3752
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9913 - loss: 0.0324 - val_accuracy: 0.8669 - val_loss: 0.5471
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.9942 - loss: 0.0213 - val_accuracy: 0.8754 - val_loss: 0.5385
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9962 - loss: 0.0138 - val_accuracy: 0.8676 - val_loss: 0.5614
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.9975 - loss: 0.0097 - val

In [47]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.872143049206776,
 'Precision': 0.8672122492080253,
 'Recall': 0.8799892847575677,
 'F1 Score': 0.8735540486637415,
 'ROC-AUC': np.float64(0.9364685901322162)}

In [126]:
optimization_results.append({
    "Experiment": "E1",
    "Enhancement": "Embediing Dimension",
    "Model": "GRU",
    'Accuracy': 0.872143049206776,
    'Precision': 0.8672122492080253,
    'Recall': 0.8799892847575677,
    'F1 Score': 0.8735540486637415,
    'ROC-AUC': 0.9364685901322162
})

In [48]:
# Hidden Layers

gru_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    GRU(
        128
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [49]:
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.5088 - loss: 0.6925 - val_accuracy: 0.5165 - val_loss: 0.6916
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.5992 - loss: 0.6180 - val_accuracy: 0.8380 - val_loss: 0.3812
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 18s 20ms/step - accuracy: 0.9000 - loss: 0.2507 - val_accuracy: 0.8789 - val_loss: 0.2870
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - accuracy: 0.9624 - loss: 0.1129 - val_accuracy: 0.8744 - val_loss: 0.3539
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.9863 - loss: 0.0467 - val_accuracy: 0.8691 - val_loss: 0.4796
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9936 - loss: 0.0240 - val_accuracy: 0.8793 - val_loss: 0.5466
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9963 - loss: 0.0142 - val_accuracy: 0.8751 - val_loss: 0.6290
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9969 - loss: 0.0109 - val_

In [50]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8682441516536703,
 'Precision': 0.8803536888643272,
 'Recall': 0.8534690597374766,
 'F1 Score': 0.8667029379760609,
 'ROC-AUC': np.float64(0.9339324325154827)}

In [127]:
optimization_results.append({
    "Experiment": "E2",
    "Enhancement": "Hidden Layers",
    "Model": "GRU",
    'Accuracy': 0.8682441516536703,
    'Precision': 0.8803536888643272,
    'Recall': 0.8534690597374766,
    'F1 Score': 0.8667029379760609,
    'ROC-AUC': 0.9339324325154827
})

In [54]:
# Dropout

gru_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    GRU(
        64
    ),

    tf.keras.layers.Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [55]:
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.5069 - loss: 0.6934 - val_accuracy: 0.5178 - val_loss: 0.6911
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - accuracy: 0.5357 - loss: 0.6735 - val_accuracy: 0.5361 - val_loss: 0.6716
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.8534 - loss: 0.3561 - val_accuracy: 0.8840 - val_loss: 0.2945
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9407 - loss: 0.1714 - val_accuracy: 0.8859 - val_loss: 0.3106
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9741 - loss: 0.0860 - val_accuracy: 0.8840 - val_loss: 0.4180
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - accuracy: 0.9874 - loss: 0.0451 - val_accuracy: 0.8762 - val_loss: 0.5382
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9925 - loss: 0.0257 - val_accuracy: 0.8751 - val_loss: 0.6064
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9945 - loss: 0.0180 - 

In [53]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step


{'Accuracy': 0.8656897015326701,
 'Precision': 0.8803561491374513,
 'Recall': 0.8475756763996786,
 'F1 Score': 0.8636549747509212,
 'ROC-AUC': np.float64(0.9379872335333586)}

In [128]:
optimization_results.append({
    "Experiment": "E3",
    "Enhancement": "DropOut 0.3",
    "Model": "GRU",
    'Accuracy': 0.8656897015326701,
    'Precision': 0.8803561491374513,
    'Recall': 0.8475756763996786,
    'F1 Score': 0.8636549747509212,
    'ROC-AUC': 0.9379872335333586
})

In [56]:
# Dropout

gru_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    GRU(
        64
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [57]:
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.5021 - loss: 0.6938 - val_accuracy: 0.5015 - val_loss: 0.6927
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.5191 - loss: 0.6850 - val_accuracy: 0.5341 - val_loss: 0.6778
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.6691 - loss: 0.5637 - val_accuracy: 0.8515 - val_loss: 0.3498
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.9055 - loss: 0.2640 - val_accuracy: 0.8895 - val_loss: 0.2794
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9565 - loss: 0.1390 - val_accuracy: 0.8884 - val_loss: 0.3306
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.9801 - loss: 0.0714 - val_accuracy: 0.8821 - val_loss: 0.4832
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9918 - loss: 0.0326 - val_accuracy: 0.8762 - val_loss: 0.5735
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9934 - loss: 0.0251 - val_a

In [58]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8716052702339339,
 'Precision': 0.878680479825518,
 'Recall': 0.8633806589874096,
 'F1 Score': 0.8709633833265775,
 'ROC-AUC': np.float64(0.9404795757862996)}

In [129]:
optimization_results.append({
    "Experiment": "E4",
    "Enhancement": "DropOut 0.5",
    "Model": "GRU",
    'Accuracy': 0.8716052702339339,
    'Precision': 0.878680479825518,
    'Recall': 0.8633806589874096,
    'F1 Score': 0.8709633833265775,
    'ROC-AUC': 0.9404795757862996
})

In [14]:
# Recurrent Dropout

gru_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    GRU(
        64,
        recurrent_dropout=0.2
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [15]:
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 254s 457ms/step - accuracy: 0.5026 - loss: 0.6934 - val_accuracy: 0.5186 - val_loss: 0.6902
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 248s 457ms/step - accuracy: 0.5333 - loss: 0.6790 - val_accuracy: 0.5360 - val_loss: 0.6750
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 248s 456ms/step - accuracy: 0.6114 - loss: 0.6074 - val_accuracy: 0.8195 - val_loss: 0.4261
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 248s 457ms/step - accuracy: 0.8811 - loss: 0.3195 - val_accuracy: 0.8735 - val_loss: 0.2987
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 246s 454ms/step - accuracy: 0.9415 - loss: 0.1823 - val_accuracy: 0.8824 - val_loss: 0.3373
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 244s 450ms/step - accuracy: 0.9687 - loss: 0.1048 - val_accuracy: 0.8778 - val_loss: 0.4286
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 244s 449ms/step - accuracy: 0.9831 - loss: 0.0591 - val_accuracy: 0.8732 - val_loss: 0.4942
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 243s 447ms/step - accuracy: 0.9898 -

In [16]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 27s 113ms/step


{'Accuracy': 0.8685130411400914,
 'Precision': 0.8721967035936233,
 'Recall': 0.8647200642914546,
 'F1 Score': 0.8684422921711057,
 'ROC-AUC': np.float64(0.9388190747221864)}

In [131]:
optimization_results.append({
    "Experiment": "E5",
    "Enhancement": "recurrent dropout",
    "Model": "GRU",
    'Accuracy': 0.8685130411400914,
    'Precision': 0.8721967035936233,
    'Recall': 0.8647200642914546,
    'F1 Score': 0.8684422921711057,
    'ROC-AUC': 0.9388190747221864
})

In [17]:
# BatchNormalization

gru_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    GRU(
        64
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [18]:
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.4994 - loss: 0.7871 - val_accuracy: 0.5042 - val_loss: 0.6917
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.5220 - loss: 0.6943 - val_accuracy: 0.5021 - val_loss: 1.2111
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.6787 - loss: 0.5471 - val_accuracy: 0.6623 - val_loss: 0.8448
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9055 - loss: 0.2470 - val_accuracy: 0.8891 - val_loss: 0.2780
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9516 - loss: 0.1444 - val_accuracy: 0.8759 - val_loss: 0.3278
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9724 - loss: 0.0877 - val_accuracy: 0.8750 - val_loss: 0.4091
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9816 - loss: 0.0618 - val_accuracy: 0.8613 - val_loss: 0.4513
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9871 - loss: 0.0435 - val_acc

In [19]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8632696961548804,
 'Precision': 0.8395,
 'Recall': 0.8995446021966247,
 'F1 Score': 0.8684857105909738,
 'ROC-AUC': np.float64(0.9328588476487021)}

In [132]:
optimization_results.append({
    "Experiment": "E6",
    "Enhancement": "BatchNormalization",
    "Model": "GRU",
    'Accuracy': 0.8632696961548804,
    'Precision': 0.8395,
    'Recall': 0.8995446021966247,
    'F1 Score': 0.8684857105909738,
    'ROC-AUC': 0.9328588476487021
})

In [20]:
# New optimizer

gru_model = Sequential([

    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    GRU(
        64
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [21]:
gru_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.5017 - loss: 0.6940 - val_accuracy: 0.5120 - val_loss: 0.6929
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.5071 - loss: 0.6931 - val_accuracy: 0.5140 - val_loss: 0.6922
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.5144 - loss: 0.6908 - val_accuracy: 0.5182 - val_loss: 0.6902
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.5377 - loss: 0.6816 - val_accuracy: 0.5368 - val_loss: 0.6773
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.5527 - loss: 0.6598 - val_accuracy: 0.5373 - val_loss: 0.6701
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.5624 - loss: 0.6418 - val_accuracy: 0.5385 - val_loss: 0.6758
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.6323 - loss: 0.5946 - val_accuracy: 0.7838 - val_loss: 0.8292
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.8765 - loss: 0.3274 - val_accu

In [23]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.891771981715515,
 'Precision': 0.8834468308014667,
 'Recall': 0.9035628181087597,
 'F1 Score': 0.8933916037610913,
 'ROC-AUC': np.float64(0.9593345704304861)}

In [133]:
optimization_results.append({
    "Experiment": "E7",
    "Enhancement": "New Optimizers Rmsprop",
    "Model": "GRU",
    'Accuracy': 0.891771981715515,
    'Precision': 0.8834468308014667,
    'Recall': 0.9035628181087597,
    'F1 Score': 0.8933916037610913,
    'ROC-AUC': 0.9593345704304861
})

In [24]:
# Learning Rate

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9604 - loss: 0.1251 - val_accuracy: 0.8968 - val_loss: 0.3037
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9694 - loss: 0.1034 - val_accuracy: 0.8958 - val_loss: 0.3196
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9766 - loss: 0.0830 - val_accuracy: 0.8931 - val_loss: 0.3727
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9835 - loss: 0.0659 - val_accuracy: 0.8911 - val_loss: 0.4228
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9880 - loss: 0.0491 - val_accuracy: 0.8872 - val_loss: 0.4773
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9925 - loss: 0.0348 - val_accuracy: 0.8855 - val_loss: 0.5664
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9952 - loss: 0.0256 - val_accuracy: 0.8816 - val_loss: 0.6338
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9968 - loss: 0.0194 - val_accu

In [25]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8732186071524604,
 'Precision': 0.8704195432819968,
 'Recall': 0.8781141173319046,
 'F1 Score': 0.8742498999866649,
 'ROC-AUC': np.float64(0.94240770485219)}

In [134]:
optimization_results.append({
    "Experiment": "E8",
    "Enhancement": "Learning rate adam 0.0001",
    "Model": "GRU",
    'Accuracy': 0.8732186071524604,
    'Precision': 0.8704195432819968,
    'Recall': 0.8781141173319046,
    'F1 Score': 0.8742498999866649,
    'ROC-AUC': 0.94240770485219
})

In [26]:
# Learning Rate

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9865 - loss: 0.0415 - val_accuracy: 0.8789 - val_loss: 0.5635
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9903 - loss: 0.0311 - val_accuracy: 0.8724 - val_loss: 0.5675
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9951 - loss: 0.0170 - val_accuracy: 0.8742 - val_loss: 0.7640
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9961 - loss: 0.0125 - val_accuracy: 0.8743 - val_loss: 0.9180
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9962 - loss: 0.0130 - val_accuracy: 0.8742 - val_loss: 0.8354
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9960 - loss: 0.0125 - val_accuracy: 0.8701 - val_loss: 0.8630
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9971 - loss: 0.0095 - val_accuracy: 0.8685 - val_loss: 1.0339
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9980 - loss: 0.0065 - val_accu

In [27]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8682441516536703,
 'Precision': 0.8774335069920483,
 'Recall': 0.8572193945888026,
 'F1 Score': 0.8672086720867209,
 'ROC-AUC': np.float64(0.9371395580793976)}

In [135]:
optimization_results.append({
    "Experiment": "E9",
    "Enhancement": "Learning rate adam 0.0005",
    "Model": "GRU",
    'Accuracy': 0.8682441516536703,
    'Precision': 0.8774335069920483,
    'Recall': 0.8572193945888026,
    'F1 Score': 0.8672086720867209,
    'ROC-AUC': 0.9371395580793976
})

In [28]:
# Learning Rate

gru_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9999 - loss: 6.7176e-04 - val_accuracy: 0.8653 - val_loss: 1.5639
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9999 - loss: 4.3498e-04 - val_accuracy: 0.8672 - val_loss: 1.7484
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9999 - loss: 2.3181e-04 - val_accuracy: 0.8672 - val_loss: 1.8708
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 1.0000 - loss: 1.0357e-04 - val_accuracy: 0.8669 - val_loss: 1.9478
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 1.0000 - loss: 6.9757e-05 - val_accuracy: 0.8675 - val_loss: 2.0125
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 1.0000 - loss: 1.1718e-04 - val_accuracy: 0.8680 - val_loss: 2.0693
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 1.0000 - loss: 7.5220e-05 - val_accuracy: 0.8661 - val_loss: 2.1694
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 1.00

In [29]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.8694541543425652,
 'Precision': 0.8632298790110469,
 'Recall': 0.8791856415751407,
 'F1 Score': 0.871134704711347,
 'ROC-AUC': np.float64(0.9241603049433635)}

In [136]:
optimization_results.append({
    "Experiment": "E10",
    "Enhancement": "Learning rate rmsprop 0.0001",
    "Model": "GRU",
    'Accuracy': 0.8694541543425652,
    'Precision': 0.8632298790110469,
    'Recall': 0.8791856415751407,
    'F1 Score': 0.871134704711347,
    'ROC-AUC': 0.9241603049433635
})

In [30]:
# Batch Size Tuning

gru_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=32
)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9991 - loss: 0.0045 - val_accuracy: 0.8700 - val_loss: 2.0461
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9990 - loss: 0.0060 - val_accuracy: 0.8595 - val_loss: 2.2028
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9989 - loss: 0.0055 - val_accuracy: 0.8605 - val_loss: 2.1540
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9988 - loss: 0.0065 - val_accuracy: 0.8622 - val_loss: 2.0326
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9990 - loss: 0.0043 - val_accuracy: 0.8644 - val_loss: 2.4222
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9994 - loss: 0.0024 - val_accuracy: 0.8691 - val_loss: 2.3951
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9992 - loss: 0.0045 - val_accuracy: 0.8711 - val_loss: 2.4887
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - accuracy: 0.9993 -

In [31]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


{'Accuracy': 0.868647485883302,
 'Precision': 0.8467539003522899,
 'Recall': 0.9014197696222878,
 'F1 Score': 0.8732321266381212,
 'ROC-AUC': np.float64(0.913353780503103)}

In [137]:
optimization_results.append({
    "Experiment": "E11",
    "Enhancement": "Batch Size 32",
    "Model": "GRU",
    'Accuracy': 0.868647485883302,
    'Precision': 0.8467539003522899,
    'Recall': 0.9014197696222878,
    'F1 Score': 0.8732321266381212,
    'ROC-AUC': 0.913353780503103
})

In [33]:
# Batch Size Tuning

gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=10,
    batch_size=16
)

Epoch 1/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 26s 11ms/step - accuracy: 0.9701 - loss: 0.0981 - val_accuracy: 0.8726 - val_loss: 0.4393
Epoch 2/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.9811 - loss: 0.0593 - val_accuracy: 0.8715 - val_loss: 0.5107
Epoch 3/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.9916 - loss: 0.0271 - val_accuracy: 0.8660 - val_loss: 0.5187
Epoch 4/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.9937 - loss: 0.0205 - val_accuracy: 0.8644 - val_loss: 0.6272
Epoch 5/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.9959 - loss: 0.0148 - val_accuracy: 0.8632 - val_loss: 1.0277
Epoch 6/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.9957 - loss: 0.0143 - val_accuracy: 0.8619 - val_loss: 1.0148
Epoch 7/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.9972 - loss: 0.0107 - val_accuracy: 0.8658 - val_loss: 0.9752
Epoch 8/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 24s 11ms/step - accuracy: 0.9971 -

In [34]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


{'Accuracy': 0.8668997042215649,
 'Precision': 0.8652463382157124,
 'Recall': 0.8703455665684436,
 'F1 Score': 0.8677884615384616,
 'ROC-AUC': np.float64(0.9342835338464647)}

In [138]:
optimization_results.append({
    "Experiment": "E12",
    "Enhancement": "Batch Size 16",
    "Model": "GRU",
    'Accuracy': 0.8668997042215649,
    'Precision': 0.8652463382157124,
    'Recall': 0.8703455665684436,
    'F1 Score': 0.8677884615384616,
    'ROC-AUC': 0.9342835338464647
})

In [35]:
# Early Stopping

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor = "val_loss",
    patience = 5,
    restore_best_weights = True
)

In [36]:
gru_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=20,
    batch_size=64,
    callbacks=[early_stopping]
)

Epoch 1/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9997 - loss: 9.0795e-04 - val_accuracy: 0.8685 - val_loss: 2.0152
Epoch 2/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 1.0000 - loss: 1.0299e-04 - val_accuracy: 0.8693 - val_loss: 2.3615
Epoch 3/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 1.0000 - loss: 3.2689e-05 - val_accuracy: 0.8683 - val_loss: 2.4795
Epoch 4/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 1.0000 - loss: 2.4675e-05 - val_accuracy: 0.8681 - val_loss: 2.6388
Epoch 5/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 1.0000 - loss: 3.0433e-05 - val_accuracy: 0.8689 - val_loss: 2.7630
Epoch 6/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 1.0000 - loss: 4.0649e-05 - val_accuracy: 0.8687 - val_loss: 2.8817


In [37]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


{'Accuracy': 0.8651519225598279,
 'Precision': 0.8693181818181818,
 'Recall': 0.8607018483793196,
 'F1 Score': 0.8649885583524027,
 'ROC-AUC': np.float64(0.9261584229071926)}

In [141]:
optimization_results.append({
    "Experiment": "E13",
    "Enhancement": "Early Stopping",
    "Model": "GRU",
    'Accuracy': 0.8651519225598279,
    'Precision': 0.8693181818181818,
    'Recall': 0.8607018483793196,
    'F1 Score': 0.8649885583524027,
    'ROC-AUC': 0.9261584229071926
})

In [38]:
# Learning Rate Scheduler

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

In [39]:
gru_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_history = gru_model.fit(
    X_train_padded,
    y_train,

    validation_data=(
        X_val_padded,
        y_val
    ),

    epochs=20,
    batch_size=64,
    callbacks=[lr_scheduler]
)

Epoch 1/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 1.0000 - loss: 1.0765e-04 - val_accuracy: 0.8689 - val_loss: 2.3279 - learning_rate: 0.0010
Epoch 2/20
540/543 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 7.7086e-05
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 1.0000 - loss: 6.1050e-05 - val_accuracy: 0.8687 - val_loss: 2.5478 - learning_rate: 0.0010
Epoch 3/20
539/543 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 2.7437e-05
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 1.0000 - loss: 4.2572e-05 - val_accuracy: 0.8695 - val_loss: 2.6148 - learning_rate: 5.0000e-04
Epoch 4/20
541/543 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 6.9014e-05
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - ac

In [40]:
gru_probabilities = gru_model.predict(
    X_test_padded
)

gru_predictions = (
    gru_probabilities.ravel() >= 0.5
).astype(int)


gru_metrics = evaluate_model(
    y_test,
    gru_predictions,
    gru_probabilities.ravel()
)

gru_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


{'Accuracy': 0.8675719279376176,
 'Precision': 0.865814696485623,
 'Recall': 0.8711492097508706,
 'F1 Score': 0.8684737615168915,
 'ROC-AUC': np.float64(0.9184960484832183)}

In [142]:
optimization_results.append({
    "Experiment": "E14",
    "Enhancement": "LR Schedule",
    "Model": "GRU",
    'Accuracy': 0.8675719279376176,
    'Precision': 0.865814696485623,
    'Recall': 0.8711492097508706,
    'F1 Score': 0.8684737615168915,
    'ROC-AUC': 0.9184960484832183
})

# Bi_LSTM

In [41]:
bi_lstm_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    
    Bidirectional(
        LSTM(64)
    ),
    
    Dense(
        32,
        activation="relu"
    ),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [42]:
bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [43]:
bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - accuracy: 0.8266 - loss: 0.3857 - val_accuracy: 0.8867 - val_loss: 0.2746
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.9275 - loss: 0.1988 - val_accuracy: 0.8886 - val_loss: 0.2891
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.9575 - loss: 0.1235 - val_accuracy: 0.8809 - val_loss: 0.4104
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.9723 - loss: 0.0825 - val_accuracy: 0.8732 - val_loss: 0.3850
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.9839 - loss: 0.0505 - val_accuracy: 0.8734 - val_loss: 0.4624
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.9906 - loss: 0.0308 - val_accuracy: 0.8685 - val_loss: 0.5364
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.9904 - loss: 0.0295 - val_accuracy: 0.8649 - val_loss: 0.5501
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.9942 - loss: 0.0199 - 

In [44]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


{'Accuracy': 0.8667652594783544,
 'Precision': 0.8685483870967742,
 'Recall': 0.8655237074738816,
 'F1 Score': 0.8670334093653562,
 'ROC-AUC': np.float64(0.9344851134409413)}

In [143]:
optimization_results.append({
    "Experiment": "E1",
    "Enhancement": "Embedding Dimension",
    "Model": "BI_LSTM",
    'Accuracy': 0.8667652594783544,
    'Precision': 0.8685483870967742,
    'Recall': 0.8655237074738816,
    'F1 Score': 0.8670334093653562,
    'ROC-AUC': 0.9344851134409413
})

In [ ]:
# Dense Layer

bi_lstm_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    Bidirectional(
        LSTM(128)
    ),
    
    Dense(
        64,
        activation="relu"
    ),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [49]:
bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 25ms/step - accuracy: 0.8229 - loss: 0.3922 - val_accuracy: 0.8896 - val_loss: 0.2895
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9174 - loss: 0.2231 - val_accuracy: 0.8883 - val_loss: 0.2819
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9490 - loss: 0.1472 - val_accuracy: 0.8735 - val_loss: 0.3370
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9641 - loss: 0.1064 - val_accuracy: 0.8711 - val_loss: 0.3535
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9787 - loss: 0.0652 - val_accuracy: 0.8701 - val_loss: 0.4491
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9810 - loss: 0.0600 - val_accuracy: 0.8199 - val_loss: 0.4810
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9798 - loss: 0.0590 - val_accuracy: 0.8790 - val_loss: 0.6092
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9904 - loss: 0.0308 - 

In [50]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.861925248722775,
 'Precision': 0.8483522142121525,
 'Recall': 0.8826680953656576,
 'F1 Score': 0.8651700144413811,
 'ROC-AUC': np.float64(0.9324204048004575)}

In [144]:
optimization_results.append({
    "Experiment": "E2",
    "Enhancement": "Hidden layer",
    "Model": "BI_LSTM",
    'Accuracy': 0.861925248722775,
    'Precision': 0.8483522142121525,
    'Recall': 0.8826680953656576,
    'F1 Score': 0.8651700144413811,
    'ROC-AUC': 0.9324204048004575
})

In [51]:
# DROP OUT 

bi_lstm_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    Bidirectional(
        LSTM(64)
    ),

    tf.keras.layers.Dropout(0.3),
    
    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [52]:
bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - accuracy: 0.8092 - loss: 0.4195 - val_accuracy: 0.8798 - val_loss: 0.3038
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9205 - loss: 0.2177 - val_accuracy: 0.8837 - val_loss: 0.2917
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9487 - loss: 0.1494 - val_accuracy: 0.8848 - val_loss: 0.3861
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9664 - loss: 0.1025 - val_accuracy: 0.8744 - val_loss: 0.3864
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9727 - loss: 0.0796 - val_accuracy: 0.8750 - val_loss: 0.4129
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9817 - loss: 0.0587 - val_accuracy: 0.8761 - val_loss: 0.6398
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9886 - loss: 0.0383 - val_accuracy: 0.8766 - val_loss: 0.6721
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9925 - loss: 0.0263 - 

In [53]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8648830330734069,
 'Precision': 0.8568812140240711,
 'Recall': 0.8773104741494776,
 'F1 Score': 0.8669755129053607,
 'ROC-AUC': np.float64(0.9317517143845622)}

In [145]:
optimization_results.append({
    "Experiment": "E3",
    "Enhancement": "DropOut 0.3",
    "Model": "BI_LSTM",
    'Accuracy': 0.8648830330734069,
    'Precision': 0.8568812140240711,
    'Recall': 0.8773104741494776,
    'F1 Score': 0.8669755129053607,
    'ROC-AUC': 0.9317517143845622
})

In [54]:
# DROP OUT 

bi_lstm_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    Bidirectional(
        LSTM(64)
    ),

    tf.keras.layers.Dropout(0.5),
    
    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [55]:
bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.8178 - loss: 0.4115 - val_accuracy: 0.8861 - val_loss: 0.2817
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9207 - loss: 0.2276 - val_accuracy: 0.8798 - val_loss: 0.2892
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9447 - loss: 0.1648 - val_accuracy: 0.8802 - val_loss: 0.3645
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9635 - loss: 0.1151 - val_accuracy: 0.8758 - val_loss: 0.3863
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9697 - loss: 0.0976 - val_accuracy: 0.8691 - val_loss: 0.4559
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9820 - loss: 0.0609 - val_accuracy: 0.8671 - val_loss: 0.4708
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9865 - loss: 0.0464 - val_accuracy: 0.8675 - val_loss: 0.5971
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9893 - loss: 0.0387 - 

In [56]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8678408174240387,
 'Precision': 0.8668623265741728,
 'Recall': 0.8703455665684436,
 'F1 Score': 0.8686004544846946,
 'ROC-AUC': np.float64(0.9350492543254114)}

In [146]:
optimization_results.append({
    "Experiment": "E4",
    "Enhancement": "DropOut 0.5",
    "Model": "BI_LSTM",
    'Accuracy': 0.8678408174240387,
    'Precision': 0.8668623265741728,
    'Recall': 0.8703455665684436,
    'F1 Score': 0.8686004544846946,
    'ROC-AUC': 0.9350492543254114
})

In [57]:
# RECURRENT DROP OUT 

bi_lstm_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    Bidirectional(
        LSTM(64 , recurrent_dropout = 0.2)
    ),

    tf.keras.layers.Dropout(0.5),
    
    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [58]:
bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 441s 804ms/step - accuracy: 0.7821 - loss: 0.4754 - val_accuracy: 0.8662 - val_loss: 0.3386
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 434s 800ms/step - accuracy: 0.8879 - loss: 0.3099 - val_accuracy: 0.8808 - val_loss: 0.3055
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 431s 793ms/step - accuracy: 0.9166 - loss: 0.2442 - val_accuracy: 0.8777 - val_loss: 0.3554
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 437s 806ms/step - accuracy: 0.9318 - loss: 0.2029 - val_accuracy: 0.8617 - val_loss: 0.3676
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 454s 836ms/step - accuracy: 0.9500 - loss: 0.1502 - val_accuracy: 0.8751 - val_loss: 0.4036
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 489s 901ms/step - accuracy: 0.9613 - loss: 0.1184 - val_accuracy: 0.8679 - val_loss: 0.4147
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 499s 919ms/step - accuracy: 0.9687 - loss: 0.0964 - val_accuracy: 0.8700 - val_loss: 0.4964
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 493s 908ms/step - accuracy: 0.9764 -

In [59]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 39s 167ms/step


{'Accuracy': 0.8591019091153536,
 'Precision': 0.8466305189775368,
 'Recall': 0.8783819983927136,
 'F1 Score': 0.8622140415461478,
 'ROC-AUC': np.float64(0.9282732010846833)}

In [147]:
optimization_results.append({
    "Experiment": "E5",
    "Enhancement": "recurrent dropout",
    "Model": "BI_LSTM",
    'Accuracy': 0.8591019091153536,
    'Precision': 0.8466305189775368,
    'Recall': 0.8783819983927136,
    'F1 Score': 0.8622140415461478,
    'ROC-AUC': 0.9282732010846833
})

In [60]:
# BatchNOrmalization

bi_lstm_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    Bidirectional(
        LSTM(64)
    ),

    tf.keras.layers.BatchNormalization(),
    
    tf.keras.layers.Dropout(0.5),
    
    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dropout(0.5),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [61]:
bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.7824 - loss: 0.4813 - val_accuracy: 0.8792 - val_loss: 0.2913
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9103 - loss: 0.2398 - val_accuracy: 0.8876 - val_loss: 0.3110
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9381 - loss: 0.1730 - val_accuracy: 0.8660 - val_loss: 0.3507
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9522 - loss: 0.1343 - val_accuracy: 0.8632 - val_loss: 0.3434
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9633 - loss: 0.1076 - val_accuracy: 0.8598 - val_loss: 0.5582
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9737 - loss: 0.0798 - val_accuracy: 0.8650 - val_loss: 0.4604
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9794 - loss: 0.0642 - val_accuracy: 0.8739 - val_loss: 0.5923
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9778 - loss: 0.0705 - 

In [62]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


{'Accuracy': 0.8670341489647755,
 'Precision': 0.8610526315789474,
 'Recall': 0.8765068309670506,
 'F1 Score': 0.8687110049117217,
 'ROC-AUC': np.float64(0.9255197380622113)}

In [148]:
optimization_results.append({
    "Experiment": "E6",
    "Enhancement": "BatchNormalization",
    "Model": "BI_LSTM",
    'Accuracy': 0.8670341489647755,
    'Precision': 0.8610526315789474,
    'Recall': 0.8765068309670506,
    'F1 Score': 0.8687110049117217,
    'ROC-AUC': 0.9255197380622113
})

In [63]:
# New Optimizers

bi_lstm_model = Sequential([
    
    Input(
        shape=(MAX_SEQUENCE_LENGTH,)
    ),
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    
    Bidirectional(
        LSTM(64)
    ),
    
    tf.keras.layers.Dropout(0.5),
    
    Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.5),
    
    Dense(
        1,
        activation="sigmoid"
    )
])

In [64]:
bi_lstm_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.7885 - loss: 0.4516 - val_accuracy: 0.8037 - val_loss: 0.6126
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.8960 - loss: 0.2864 - val_accuracy: 0.8844 - val_loss: 0.3201
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9203 - loss: 0.2303 - val_accuracy: 0.8814 - val_loss: 0.2915
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9337 - loss: 0.1917 - val_accuracy: 0.8829 - val_loss: 0.2953
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9480 - loss: 0.1586 - val_accuracy: 0.8886 - val_loss: 0.3370
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9592 - loss: 0.1314 - val_accuracy: 0.8872 - val_loss: 0.3777
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9684 - loss: 0.1062 - val_accuracy: 0.8758 - val_loss: 0.3706
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9744 - loss: 0.0867 - 

In [65]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


{'Accuracy': 0.8763108362463028,
 'Precision': 0.878396556362658,
 'Recall': 0.8746316635413877,
 'F1 Score': 0.876510067114094,
 'ROC-AUC': np.float64(0.9435704026494558)}

In [149]:
optimization_results.append({
    "Experiment": "E7",
    "Enhancement": "New Optimizers Rmsprop",
    "Model": "BI_LSTM",
    'Accuracy': 0.8670341489647755,
    'Precision': 0.8610526315789474,
    'Recall': 0.8765068309670506,
    'F1 Score': 0.8687110049117217,
    'ROC-AUC': 0.9255197380622113
})

In [66]:
# Learning rate

bi_lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.9954 - loss: 0.0189 - val_accuracy: 0.8778 - val_loss: 0.6991
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9982 - loss: 0.0100 - val_accuracy: 0.8782 - val_loss: 0.8570
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9987 - loss: 0.0063 - val_accuracy: 0.8769 - val_loss: 0.8950
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9992 - loss: 0.0050 - val_accuracy: 0.8785 - val_loss: 1.0034
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9993 - loss: 0.0046 - val_accuracy: 0.8787 - val_loss: 1.0011
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9997 - loss: 0.0023 - val_accuracy: 0.8751 - val_loss: 1.0220
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9997 - loss: 0.0019 - val_accuracy: 0.8751 - val_loss: 1.1022
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9999 - loss: 8.6206e-0

In [67]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8736219413820919,
 'Precision': 0.8730964467005076,
 'Recall': 0.8754353067238146,
 'F1 Score': 0.8742643124665597,
 'ROC-AUC': np.float64(0.937514989228723)}

In [150]:
optimization_results.append({
    "Experiment": "E8",
    "Enhancement": "Learning Rate adam 0.0001",
    "Model": "BI_LSTM",
    'Accuracy': 0.8736219413820919,
    'Precision': 0.8730964467005076,
    'Recall': 0.8754353067238146,
    'F1 Score': 0.8742643124665597,
    'ROC-AUC': 0.937514989228723
})

In [68]:
# Learning rate

bi_lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.9940 - loss: 0.0219 - val_accuracy: 0.8691 - val_loss: 0.7103
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9952 - loss: 0.0172 - val_accuracy: 0.8723 - val_loss: 0.7460
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9967 - loss: 0.0128 - val_accuracy: 0.8754 - val_loss: 0.9589
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9955 - loss: 0.0182 - val_accuracy: 0.8759 - val_loss: 0.7928
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9979 - loss: 0.0091 - val_accuracy: 0.8681 - val_loss: 0.9170
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9977 - loss: 0.0108 - val_accuracy: 0.8730 - val_loss: 0.8686
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9987 - loss: 0.0050 - val_accuracy: 0.8739 - val_loss: 0.9325
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9977 - loss: 0.0087 - 

In [69]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8760419467598817,
 'Precision': 0.8816182459951126,
 'Recall': 0.8698098044468257,
 'F1 Score': 0.8756742179072277,
 'ROC-AUC': np.float64(0.9405810886093429)}

In [151]:
optimization_results.append({
    "Experiment": "E9",
    "Enhancement": "Learning Rate adam 0.0005",
    "Model": "BI_LSTM",
    'Accuracy': 0.8760419467598817,
    'Precision': 0.8816182459951126,
    'Recall': 0.8698098044468257,
    'F1 Score': 0.8756742179072277,
    'ROC-AUC': 0.9405810886093429
})

In [70]:
# Learning rate

bi_lstm_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate = 0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.9993 - loss: 0.0036 - val_accuracy: 0.8730 - val_loss: 1.1637
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9996 - loss: 0.0018 - val_accuracy: 0.8785 - val_loss: 1.4933
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9997 - loss: 0.0013 - val_accuracy: 0.8720 - val_loss: 1.5031
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9996 - loss: 0.0027 - val_accuracy: 0.8793 - val_loss: 1.6876
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9998 - loss: 7.1425e-04 - val_accuracy: 0.8771 - val_loss: 1.8008
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9999 - loss: 7.5554e-04 - val_accuracy: 0.8750 - val_loss: 1.7491
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9999 - loss: 4.6938e-04 - val_accuracy: 0.8738 - val_loss: 1.9107
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9997 - los

In [71]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8733530518956709,
 'Precision': 0.8774682174736272,
 'Recall': 0.8690061612643986,
 'F1 Score': 0.8732166890982503,
 'ROC-AUC': np.float64(0.9300122950538167)}

In [152]:
optimization_results.append({
    "Experiment": "E10",
    "Enhancement": "Learning Rate Rmsprop 0.0001",
    "Model": "BI_LSTM",
    'Accuracy': 0.8733530518956709,
    'Precision': 0.8774682174736272,
    'Recall': 0.8690061612643986,
    'F1 Score': 0.8732166890982503,
    'ROC-AUC': 0.9300122950538167
})

In [73]:
# Batch Size 

bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=32
)

Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 23s 18ms/step - accuracy: 0.9831 - loss: 0.0559 - val_accuracy: 0.8693 - val_loss: 0.7311
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9856 - loss: 0.0449 - val_accuracy: 0.8484 - val_loss: 0.4872
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9890 - loss: 0.0380 - val_accuracy: 0.8632 - val_loss: 0.6739
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9934 - loss: 0.0234 - val_accuracy: 0.8489 - val_loss: 0.7264
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9949 - loss: 0.0206 - val_accuracy: 0.8587 - val_loss: 0.7387
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9951 - loss: 0.0183 - val_accuracy: 0.8658 - val_loss: 0.8704
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9964 - loss: 0.0141 - val_accuracy: 0.8726 - val_loss: 0.8575
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9943 -

In [74]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8634041408980909,
 'Precision': 0.8785176929506826,
 'Recall': 0.8446289847307795,
 'F1 Score': 0.8612400983337886,
 'ROC-AUC': np.float64(0.935813131088555)}

In [153]:
optimization_results.append({
    "Experiment": "E11",
    "Enhancement": "Batch Size 32",
    "Model": "BI_LSTM",
    'Accuracy': 0.8634041408980909,
    'Precision': 0.8785176929506826,
    'Recall': 0.8446289847307795,
    'F1 Score': 0.8612400983337886,
    'ROC-AUC': 0.935813131088555
})

In [75]:
# Batch Size 


bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=16
)

Epoch 1/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 39s 17ms/step - accuracy: 0.9939 - loss: 0.0221 - val_accuracy: 0.8641 - val_loss: 0.8452
Epoch 2/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 37s 17ms/step - accuracy: 0.9955 - loss: 0.0159 - val_accuracy: 0.8598 - val_loss: 0.8112
Epoch 3/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 37s 17ms/step - accuracy: 0.9972 - loss: 0.0106 - val_accuracy: 0.8572 - val_loss: 0.9069
Epoch 4/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 37s 17ms/step - accuracy: 0.9976 - loss: 0.0089 - val_accuracy: 0.8394 - val_loss: 0.9383
Epoch 5/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 37s 17ms/step - accuracy: 0.9967 - loss: 0.0119 - val_accuracy: 0.8649 - val_loss: 0.9509
Epoch 6/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 38s 17ms/step - accuracy: 0.9983 - loss: 0.0068 - val_accuracy: 0.8679 - val_loss: 0.9465
Epoch 7/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 37s 17ms/step - accuracy: 0.9983 - loss: 0.0055 - val_accuracy: 0.8720 - val_loss: 1.1636
Epoch 8/10
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 37s 17ms/step - accuracy: 0.9978 -

In [76]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8650174778166174,
 'Precision': 0.8524928958925342,
 'Recall': 0.8840075006697027,
 'F1 Score': 0.8679642293529721,
 'ROC-AUC': np.float64(0.9326804048799904)}

In [154]:
optimization_results.append({
    "Experiment": "E12",
    "Enhancement": "Batch Size 16",
    "Model": "BI_LSTM",
    'Accuracy': 0.8650174778166174,
    'Precision': 0.8524928958925342,
    'Recall': 0.8840075006697027,
    'F1 Score': 0.8679642293529721,
    'ROC-AUC': 0.9326804048799904
})

In [77]:
# Early Stopping

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor = "val_loss",
    patience = 5,
    restore_best_weights = True
)

In [78]:
# Early Stopping

bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=20,
    batch_size=64,
    callbacks=[early_stopping]
)

Epoch 1/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 20ms/step - accuracy: 0.9999 - loss: 6.7576e-04 - val_accuracy: 0.8681 - val_loss: 1.5992
Epoch 2/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.9999 - loss: 1.9289e-04 - val_accuracy: 0.8653 - val_loss: 1.7453
Epoch 3/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 1.0000 - loss: 8.9776e-05 - val_accuracy: 0.8679 - val_loss: 1.8764
Epoch 4/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 1.0000 - loss: 8.3964e-05 - val_accuracy: 0.8689 - val_loss: 1.9837
Epoch 5/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 1.0000 - loss: 4.9010e-05 - val_accuracy: 0.8685 - val_loss: 2.0574
Epoch 6/20
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 1.0000 - loss: 1.2004e-04 - val_accuracy: 0.8687 - val_loss: 2.1086


In [79]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


{'Accuracy': 0.8709330465178812,
 'Precision': 0.86845601913367,
 'Recall': 0.8754353067238146,
 'F1 Score': 0.871931696905016,
 'ROC-AUC': np.float64(0.9330897820908677)}

In [155]:
optimization_results.append({
    "Experiment": "E13",
    "Enhancement": "Early Stopping",
    "Model": "BI_LSTM",
    'Accuracy': 0.8709330465178812,
    'Precision': 0.86845601913367,
    'Recall': 0.8754353067238146,
    'F1 Score': 0.871931696905016,
    'ROC-AUC': 0.9330897820908677
})

In [80]:
# LR Schedule

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

In [81]:


bi_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bi_lstm_history = bi_lstm_model.fit(
    X_train_padded,
    y_train,
    
    validation_data=(
        X_val_padded,
        y_val
    ),
    
    epochs=10,
    batch_size=64,
    callbacks=[lr_scheduler]
)

Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 19ms/step - accuracy: 0.9999 - loss: 3.2580e-04 - val_accuracy: 0.8704 - val_loss: 1.7947 - learning_rate: 5.0000e-04
Epoch 2/10
540/543 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 5.9548e-05
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 1.0000 - loss: 5.8120e-05 - val_accuracy: 0.8679 - val_loss: 1.8909 - learning_rate: 5.0000e-04
Epoch 3/10
542/543 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 1.6164e-04
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 1.0000 - loss: 1.1697e-04 - val_accuracy: 0.8685 - val_loss: 1.9087 - learning_rate: 2.5000e-04
Epoch 4/10
541/543 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 8.4034e-05
Epoch 4: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 19m

In [82]:
bi_lstm_probabilities = bi_lstm_model.predict(
    X_test_padded
)

bi_lstm_predictions = (
    bi_lstm_probabilities.ravel() >= 0.5
).astype(int)

bi_lstm_metrics = evaluate_model(
    y_test,
    bi_lstm_predictions,
    bi_lstm_probabilities.ravel()
)

bi_lstm_metrics

233/233 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


{'Accuracy': 0.8702608228018285,
 'Precision': 0.8726440495422725,
 'Recall': 0.8682025180819716,
 'F1 Score': 0.8704176178326843,
 'ROC-AUC': np.float64(0.9292137130520257)}

In [156]:
optimization_results.append({
    "Experiment": "E14",
    "Enhancement": "LR Schedule",
    "Model": "BI_LSTM",
    'Accuracy': 0.8702608228018285,
    'Precision': 0.8726440495422725,
    'Recall': 0.8682025180819716,
    'F1 Score': 0.8704176178326843,
    'ROC-AUC': 0.9292137130520257
})

In [157]:
optimization_results

[{'Experiment': 'E1',
  'Enhancement': 'DropOut 0.3',
  'Model': 'ANN',
  'Accuracy': 0.8827641839204087,
  'Precision': 0.9001398601398601,
  'Recall': 0.8620412536833646,
  'F1 Score': 0.8806787082649151,
  'ROC-AUC': 0.953827102116188},
 {'Experiment': 'E2',
  'Enhancement': 'DropOut 0.5',
  'Model': 'ANN',
  'Accuracy': 0.8822264049475665,
  'Precision': 0.8897680763983629,
  'Recall': 0.8735601392981516,
  'F1 Score': 0.8815896188158961,
  'ROC-AUC': 0.9551873667147117},
 {'Experiment': 'E3',
  'Enhancement': 'BatchNormalization',
  'Model': 'ANN',
  'Accuracy': 0.8822264049475665,
  'Precision': 0.8897680763983629,
  'Recall': 0.8735601392981516,
  'F1 Score': 0.8815896188158961,
  'ROC-AUC': 0.9544096801586897},
 {'Experiment': 'E4',
  'Enhancement': 'Optimizers RMSprop',
  'Model': 'ANN',
  'Accuracy': 0.8822264049475665,
  'Precision': 0.8897680763983629,
  'Recall': 0.8735601392981516,
  'F1 Score': 0.8815896188158961,
  'ROC-AUC': 0.9543755533406866},
 {'Experiment': 'E5',
 

In [158]:
optimization_results_df = pd.DataFrame(
    optimization_results
)

In [163]:
optimization_results_df.head(32)

,Experiment,Enhancement,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,E1,DropOut 0.3,ANN,0.882764,0.900140,0.862041,0.880679,0.953827
1,E2,DropOut 0.5,ANN,0.882226,0.889768,0.873560,0.881590,0.955187
2,E3,BatchNormalization,ANN,0.882226,0.889768,0.873560,0.881590,0.954410
3,E4,Optimizers RMSprop,ANN,0.882226,0.889768,0.873560,0.881590,0.954376
4,E5,Learning Rate Rmsprop(0.0001),ANN,0.904679,0.904494,0.905706,0.905100,0.967430
5,E6,Learning Rate Adam(0.0001),ANN,0.896209,0.897877,0.894991,0.896431,0.963490
6,E7,Learning Rate Adam(0.00025),ANN,0.888142,0.890234,0.886418,0.888322,0.956508
7,E8,Learning Rate Rmsprop(0.00025),ANN,0.885588,0.882431,0.890705,0.886548,0.955328
8,E9,Batch Size 32,ANN,0.904948,0.905848,0.904634,0.905241,0.967277
9,E10,Early Stopping,ANN,0.905216,0.900741,0.911599,0.906138,0.967254


In [162]:
optimization_results_df.tail(40)

,Experiment,Enhancement,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
32,E5,recurrent dropout,LSTM,0.867437,0.851009,0.892044,0.871044,0.933252
33,E6,Batchnormalization,LSTM,0.847136,0.917363,0.764265,0.833845,0.932618
34,E7,New Optimizers rmsprop,LSTM,0.874025,0.832224,0.938119,0.882005,0.933534
35,E8,learning rate adam 0.0001,LSTM,0.876714,0.883234,0.869274,0.876198,0.944330
36,E9,learning rate adam 0.0005,LSTM,0.869589,0.863840,0.878650,0.871182,0.939575
37,E10,learning rate rmsprop 0.0001,LSTM,0.871336,0.875948,0.866327,0.871111,0.935802
38,E11,Batch size 32,LSTM,0.873219,0.873995,0.873292,0.873643,0.920754
39,E12,Batch size 16,LSTM,0.870261,0.875475,0.864452,0.869929,0.926957
40,E13,Early stopping,LSTM,0.869992,0.875203,0.864184,0.869659,0.935676
41,E14,LR Schedule,LSTM,0.872681,0.873659,0.872489,0.873073,0.923422


In [164]:
optimization_results_df.duplicated().sum()

np.int64(6)

In [165]:
optimization_results_df.duplicated()

0     False
1     False
2     False
3     False
4     False
      ...  
67    False
68    False
69    False
70    False
71    False
Length: 72, dtype: bool

In [168]:
optimization_results_df = optimization_results_df.drop_duplicates().reset_index(drop=True)


In [169]:
optimization_results_df

,Experiment,Enhancement,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,E1,DropOut 0.3,ANN,0.882764,0.900140,0.862041,0.880679,0.953827
1,E2,DropOut 0.5,ANN,0.882226,0.889768,0.873560,0.881590,0.955187
2,E3,BatchNormalization,ANN,0.882226,0.889768,0.873560,0.881590,0.954410
3,E4,Optimizers RMSprop,ANN,0.882226,0.889768,0.873560,0.881590,0.954376
4,E5,Learning Rate Rmsprop(0.0001),ANN,0.904679,0.904494,0.905706,0.905100,0.967430
...,...,...,...,...,...,...,...,...
61,E10,Learning Rate Rmsprop 0.0001,BI_LSTM,0.873353,0.877468,0.869006,0.873217,0.930012
62,E11,Batch Size 32,BI_LSTM,0.863404,0.878518,0.844629,0.861240,0.935813
63,E12,Batch Size 16,BI_LSTM,0.865017,0.852493,0.884008,0.867964,0.932680
64,E13,Early Stopping,BI_LSTM,0.870933,0.868456,0.875435,0.871932,0.933090


In [171]:
optimization_results_df.to_csv(
    "/kaggle/working/optimization_results.csv",
    index=False
)